In [1]:

from pathlib import Path
import json
import random
import numpy as np

# Project paths
PROJECT_ROOT = Path.cwd().parent

EXPERIMENT_NAME = "experiment_c2_icbhi_fraiwan_high_capacity"
OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / EXPERIMENT_NAME
)

# Create a separate output directory; do not overwrite Experiment B or C1.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Original eight-class disease mapping
CLASS_NAMES = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI",
]

CLASS_TO_IDX = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

# Audio preprocessing configuration
SAMPLE_RATE = 4000
DURATION_SECONDS = 5
NUM_SAMPLES = SAMPLE_RATE * DURATION_SECONDS
N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64
N_MFCC = 13
N_CHROMA = 12

# Initial model capacity target.
# The final count will be measured from the actual architecture.
TARGET_MIN_PARAMETERS = 50000

config = {
    "experiment_name": EXPERIMENT_NAME,
    "seed": SEED,
    "class_names": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
    "sample_rate": SAMPLE_RATE,
    "duration_seconds": DURATION_SECONDS,
    "num_samples": NUM_SAMPLES,
    "n_fft": N_FFT,
    "hop_length": HOP_LENGTH,
    "n_mels": N_MELS,
    "n_mfcc": N_MFCC,
    "n_chroma": N_CHROMA,
    "target_min_parameters": TARGET_MIN_PARAMETERS,
}

print("Project root:", PROJECT_ROOT)
print("Experiment output:", OUTPUT_ROOT)
print("Number of classes:", len(CLASS_NAMES))
print("Target minimum CNN parameters:", TARGET_MIN_PARAMETERS)
print("Output directory exists:", OUTPUT_ROOT.exists())
print("Existing experiment artifacts will not be overwritten.")

print("\nClass mapping:")
print(json.dumps(CLASS_TO_IDX, indent=2))


Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Experiment output: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity
Number of classes: 8
Target minimum CNN parameters: 50000
Output directory exists: True
Existing experiment artifacts will not be overwritten.

Class mapping:
{
  "Asthma": 0,
  "Bronchiectasis": 1,
  "Bronchiolitis": 2,
  "COPD": 3,
  "Healthy": 4,
  "LRTI": 5,
  "Pneumonia": 6,
  "URTI": 7
}


In [2]:

from pathlib import Path
import pandas as pd

BASE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
)

# Expected Experiment B manifests
manifest_candidates = [
    BASE_DIR / "experiment_b_train_manifest.csv",
    BASE_DIR / "experiment_b_icbhi_test_manifest.csv",
    BASE_DIR / "experiment_b_fraiwan_test_manifest.csv",
]

manifest_tables = {}

print("=== MANIFEST DISCOVERY ===")

for path in manifest_candidates:
    print(f"\nFile: {path}")
    if not path.exists():
        print("STATUS: NOT FOUND")
        continue

    df = pd.read_csv(path)
    manifest_tables[path.name] = df

    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    display(df.head(3))

print("\n=== LABEL COLUMN CANDIDATES ===")

for name, df in manifest_tables.items():
    possible = [
        col for col in df.columns
        if any(term in col.lower() for term in ("label", "class", "diagnos", "disease"))
    ]
    print(f"\n{name}: {possible}")

    for col in possible:
        print(f"  {col}:")
        print(df[col].value_counts(dropna=False).head(12).to_dict())

print("\n=== AUDIO PATH COLUMN CANDIDATES ===")

for name, df in manifest_tables.items():
    possible = [
        col for col in df.columns
        if any(term in col.lower() for term in ("path", "file", "audio", "wav"))
    ]
    print(f"\n{name}: {possible}")

    for col in possible:
        non_null = df[col].dropna()
        if len(non_null):
            print(f"  {col} examples:", non_null.astype(str).head(3).tolist())

print("\nRead-only manifest audit complete.")


=== MANIFEST DISCOVERY ===

File: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_train_manifest.csv
Rows: 632
Columns: ['source_dataset', 'source_recording_id', 'patient_id', 'diagnosis', 'split', 'recording_path']


,source_dataset,source_recording_id,patient_id,diagnosis,split,recording_path
0,ICBHI,103_2b2_Ar_mc_LittC2SE,ICBHI_103,Asthma,train,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
1,ICBHI,105_1b1_Tc_sc_Meditron,ICBHI_105,URTI,train,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
2,ICBHI,106_2b1_Pl_mc_LittC2SE,ICBHI_106,COPD,train,/Users/vs/Sem 5/Respiratory-Disease-Classifica...



File: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_icbhi_test_manifest.csv
Rows: 394
Columns: ['source_dataset', 'source_recording_id', 'patient_id', 'diagnosis', 'split', 'recording_path']


,source_dataset,source_recording_id,patient_id,diagnosis,split,recording_path
0,ICBHI,101_1b1_Al_sc_Meditron,ICBHI_101,URTI,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
1,ICBHI,101_1b1_Pr_sc_Meditron,ICBHI_101,URTI,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
2,ICBHI,102_1b1_Ar_sc_Meditron,ICBHI_102,Healthy,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...



File: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_fraiwan_test_manifest.csv
Rows: 28
Columns: ['source_dataset', 'source_recording_id', 'patient_id', 'diagnosis', 'split', 'recording_path']


,source_dataset,source_recording_id,patient_id,diagnosis,split,recording_path
0,Fraiwan,"BP15_Asthma,E W,P R U,49,F.wav",FRAIWAN_15,Asthma,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
1,Fraiwan,"BP27_asthma,E W,P R M,53,F.wav",FRAIWAN_27,Asthma,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
2,Fraiwan,"BP38_Asthma,E W,P R M,38,M.wav",FRAIWAN_38,Asthma,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...



=== LABEL COLUMN CANDIDATES ===

experiment_b_train_manifest.csv: ['diagnosis']
  diagnosis:
{'COPD': 452, 'Asthma': 75, 'Pneumonia': 48, 'Healthy': 18, 'URTI': 16, 'Bronchiectasis': 14, 'Bronchiolitis': 7, 'LRTI': 2}

experiment_b_icbhi_test_manifest.csv: ['diagnosis']
  diagnosis:
{'COPD': 362, 'Healthy': 17, 'URTI': 7, 'Bronchiolitis': 6, 'Bronchiectasis': 2}

experiment_b_fraiwan_test_manifest.csv: ['diagnosis']
  diagnosis:
{'Asthma': 19, 'COPD': 6, 'Pneumonia': 3}

=== AUDIO PATH COLUMN CANDIDATES ===

experiment_b_train_manifest.csv: ['recording_path']
  recording_path examples: ['/Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/103_2b2_Ar_mc_LittC2SE.wav', '/Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/105_1b1_Tc_sc_Meditron.wav', '/Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/106_2b1_Pl_mc_LittC2SE.wav']

experiment_b_icbhi_test_manifest.csv: ['recording_path']
  recording_path examples: ['/Users/vs/Sem 5/Respiratory-Di

In [3]:

import pandas as pd
from pathlib import Path

train_path = BASE_DIR / "experiment_b_train_manifest.csv"
icbhi_test_path = BASE_DIR / "experiment_b_icbhi_test_manifest.csv"
fraiwan_test_path = BASE_DIR / "experiment_b_fraiwan_test_manifest.csv"

train_df = pd.read_csv(train_path)
icbhi_test_df = pd.read_csv(icbhi_test_path)
fraiwan_test_df = pd.read_csv(fraiwan_test_path)

# Combine the two external test manifests for integrity checks.
test_df = pd.concat(
    [icbhi_test_df, fraiwan_test_df],
    ignore_index=True
)

required_columns = [
    "source_dataset",
    "source_recording_id",
    "patient_id",
    "diagnosis",
    "split",
    "recording_path",
]

for name, df in [
    ("train", train_df),
    ("ICBHI test", icbhi_test_df),
    ("Fraiwan test", fraiwan_test_df),
]:
    missing_columns = sorted(set(required_columns) - set(df.columns))
    print(f"\n=== {name.upper()} ===")
    print("Rows:", len(df))
    print("Missing required columns:", missing_columns)
    print("Duplicate recording IDs:",
          int(df["source_recording_id"].duplicated().sum()))
    print("Missing labels:", int(df["diagnosis"].isna().sum()))
    print("Unexpected labels:",
          sorted(set(df["diagnosis"].dropna()) - set(CLASS_NAMES)))

    missing_audio = [
        p for p in df["recording_path"].dropna().astype(str)
        if not Path(p).is_file()
    ]
    print("Missing audio files:", len(missing_audio))
    if missing_audio:
        print("Examples:", missing_audio[:5])

train_patients = set(train_df["patient_id"].dropna())
test_patients = set(test_df["patient_id"].dropna())

print("\n=== TRAIN / TEST SEPARATION ===")
print("Train patients:", len(train_patients))
print("Test patients:", len(test_patients))
print("Overlapping patient IDs:",
      sorted(train_patients & test_patients))

train_recordings = set(train_df["recording_path"].dropna())
test_recordings = set(test_df["recording_path"].dropna())

print("Overlapping recording paths:",
      len(train_recordings & test_recordings))

print("\n=== CLASS SUPPORT ===")
support = pd.concat([
    train_df.assign(manifest_split="train"),
    test_df.assign(manifest_split="test"),
]).groupby(
    ["manifest_split", "source_dataset", "diagnosis"]
).size().unstack(fill_value=0)

display(support)

print("\nRead-only integrity audit complete.")



=== TRAIN ===
Rows: 632
Missing required columns: []
Duplicate recording IDs: 0
Missing labels: 0
Unexpected labels: []
Missing audio files: 0

=== ICBHI TEST ===
Rows: 394
Missing required columns: []
Duplicate recording IDs: 0
Missing labels: 0
Unexpected labels: []
Missing audio files: 0

=== FRAIWAN TEST ===
Rows: 28
Missing required columns: []
Duplicate recording IDs: 0
Missing labels: 0
Unexpected labels: []
Missing audio files: 0

=== TRAIN / TEST SEPARATION ===
Train patients: 113
Test patients: 59
Overlapping patient IDs: []
Overlapping recording paths: 0

=== CLASS SUPPORT ===


diagnosis                      Asthma  Bronchiectasis  Bronchiolitis  COPD  \
manifest_split source_dataset                                                
test           Fraiwan             19               0              0     6   
               ICBHI                0               2              6   362   
train          Fraiwan             74               0              0    21   
               ICBHI                1              14              7   431   

diagnosis                      Healthy  LRTI  Pneumonia  URTI  
manifest_split source_dataset                                  
test           Fraiwan               0     0          3     0  
               ICBHI                17     0          0     7  
train          Fraiwan               0     0         12     0  
               ICBHI                18     2         36    16


Read-only integrity audit complete.


In [4]:

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Reload the verified source manifest.
train_df = pd.read_csv(BASE_DIR / "experiment_b_train_manifest.csv")
icbhi_test_df = pd.read_csv(BASE_DIR / "experiment_b_icbhi_test_manifest.csv")
fraiwan_test_df = pd.read_csv(BASE_DIR / "experiment_b_fraiwan_test_manifest.csv")

# Group by dataset + patient ID to prevent accidental cross-source ID collisions.
train_df = train_df.copy()
train_df["group_id"] = (
    train_df["source_dataset"].astype(str)
    + "::"
    + train_df["patient_id"].astype(str)
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED,
)

train_idx, val_idx = next(
    splitter.split(train_df, groups=train_df["group_id"])
)

c2_train_df = train_df.iloc[train_idx].drop(columns=["group_id"]).reset_index(drop=True)
c2_val_df = train_df.iloc[val_idx].drop(columns=["group_id"]).reset_index(drop=True)

# Confirm patient separation.
train_groups = set(
    train_df.iloc[train_idx]["group_id"]
)
val_groups = set(
    train_df.iloc[val_idx]["group_id"]
)

assert train_groups.isdisjoint(val_groups)

# Save only new C2 manifests.
c2_train_path = OUTPUT_ROOT / "c2_train_manifest.csv"
c2_val_path = OUTPUT_ROOT / "c2_validation_manifest.csv"
c2_icbhi_test_path = OUTPUT_ROOT / "c2_icbhi_test_manifest.csv"
c2_fraiwan_test_path = OUTPUT_ROOT / "c2_fraiwan_test_manifest.csv"

c2_train_df.to_csv(c2_train_path, index=False)
c2_val_df.to_csv(c2_val_path, index=False)
icbhi_test_df.to_csv(c2_icbhi_test_path, index=False)
fraiwan_test_df.to_csv(c2_fraiwan_test_path, index=False)

print("=== C2 SPLIT SUMMARY ===")
print("Training recordings:", len(c2_train_df))
print("Validation recordings:", len(c2_val_df))
print("ICBHI test recordings:", len(icbhi_test_df))
print("Fraiwan test recordings:", len(fraiwan_test_df))
print("Training patients:", c2_train_df["patient_id"].nunique())
print("Validation patients:", c2_val_df["patient_id"].nunique())
print("Patient overlap:", len(train_groups & val_groups))

print("\n=== VALIDATION CLASS SUPPORT ===")
display(
    c2_val_df.groupby(["source_dataset", "diagnosis"])
    .size()
    .unstack(fill_value=0)
)

print("\n=== SAVED MANIFESTS ===")
for path in [
    c2_train_path,
    c2_val_path,
    c2_icbhi_test_path,
    c2_fraiwan_test_path,
]:
    print(path, "| exists:", path.exists())

print("\nOriginal Experiment B manifests were not modified.")


=== C2 SPLIT SUMMARY ===
Training recordings: 530
Validation recordings: 102
ICBHI test recordings: 394
Fraiwan test recordings: 28
Training patients: 90
Validation patients: 23
Patient overlap: 0

=== VALIDATION CLASS SUPPORT ===


diagnosis,Asthma,COPD,Healthy,LRTI,Pneumonia
source_dataset,,,,,
Fraiwan,9,6,0,0,6
ICBHI,1,71,7,2,0



=== SAVED MANIFESTS ===
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/c2_train_manifest.csv | exists: True
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/c2_validation_manifest.csv | exists: True
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/c2_icbhi_test_manifest.csv | exists: True
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/c2_fraiwan_test_manifest.csv | exists: True

Original Experiment B manifests were not modified.


In [5]:

import sys
import platform
import importlib.util

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Machine:", platform.machine())

for package in ["torch", "torchaudio", "tensorflow", "librosa", "soundfile", "sklearn"]:
    available = importlib.util.find_spec(package) is not None
    print(f"{package}: {'installed' if available else 'not installed'}")

if importlib.util.find_spec("torch") is not None:
    import torch

    print("\n=== PYTORCH COMPUTE ===")
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("MPS available:", hasattr(torch.backends, "mps") and torch.backends.mps.is_available())

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("Selected accelerator: Apple Metal (MPS)")
    elif torch.cuda.is_available():
        print("Selected accelerator: NVIDIA CUDA")
    else:
        print("Selected accelerator: CPU")

print("\nEnvironment check complete. No model training performed.")


Python: 3.14.7
Platform: macOS-27.0.1-arm64-arm-64bit-Mach-O
Machine: arm64
torch: installed
torchaudio: not installed
tensorflow: not installed
librosa: installed
soundfile: installed
sklearn: installed

=== PYTORCH COMPUTE ===
PyTorch version: 2.14.1
CUDA available: False
MPS available: True
Selected accelerator: Apple Metal (MPS)

Environment check complete. No model training performed.


In [6]:

import numpy as np
import librosa
import soundfile as sf
import torch

# Use one real recording from the new training manifest.
sample_path = Path(c2_train_df.iloc[0]["recording_path"])

print("Sample file:", sample_path.name)
print("File exists:", sample_path.is_file())

# Inspect original WAV metadata.
info = sf.info(str(sample_path))
print("\n=== SOURCE AUDIO ===")
print("Sample rate:", info.samplerate)
print("Channels:", info.channels)
print("Duration (seconds):", round(info.duration, 3))

# Load and resample to the experiment's target rate.
audio, source_sr = librosa.load(
    str(sample_path),
    sr=SAMPLE_RATE,
    mono=True,
)

# Match the model's fixed 5-second input duration.
if len(audio) < NUM_SAMPLES:
    audio = np.pad(audio, (0, NUM_SAMPLES - len(audio)))
else:
    audio = audio[:NUM_SAMPLES]

# Generate a log-Mel spectrogram.
mel = librosa.feature.melspectrogram(
    y=audio.astype(np.float32),
    sr=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    fmin=0,
    fmax=SAMPLE_RATE / 2,
    power=2.0,
)

log_mel = librosa.power_to_db(mel, ref=1.0).astype(np.float32)

print("\n=== PREPROCESSING CHECK ===")
print("Loaded sample rate:", source_sr)
print("Waveform shape:", audio.shape)
print("Log-Mel shape:", log_mel.shape)
print("Log-Mel finite:", bool(np.isfinite(log_mel).all()))
print("Log-Mel min/max:", float(log_mel.min()), float(log_mel.max()))
print("PyTorch device:", "mps" if torch.backends.mps.is_available() else "cpu")

assert audio.shape == (NUM_SAMPLES,)
assert log_mel.shape[0] == N_MELS
assert np.isfinite(log_mel).all()

print("\nPreprocessing check passed. No dataset cache was created.")


Sample file: 105_1b1_Tc_sc_Meditron.wav
File exists: True

=== SOURCE AUDIO ===
Sample rate: 44100
Channels: 1
Duration (seconds): 20.0

=== PREPROCESSING CHECK ===
Loaded sample rate: 4000
Waveform shape: (20000,)
Log-Mel shape: (64, 157)
Log-Mel finite: True
Log-Mel min/max: -75.26664733886719 4.733355522155762
PyTorch device: mps

Preprocessing check passed. No dataset cache was created.


In [7]:

import json
import time
import numpy as np
import pandas as pd
import librosa
from pathlib import Path

FEATURE_ROOT = OUTPUT_ROOT / "features_logmel_v1"
FEATURE_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS = {
    "train": c2_train_df,
    "validation": c2_val_df,
    "icbhi_test": icbhi_test_df,
    "fraiwan_test": fraiwan_test_df,
}

def extract_logmel(recording_path):
    audio, _ = librosa.load(
        str(recording_path),
        sr=SAMPLE_RATE,
        mono=True,
    )
    audio = np.asarray(audio, dtype=np.float32)

    # Select a deterministic central 5-second crop, or pad short recordings.
    if len(audio) > NUM_SAMPLES:
        start = (len(audio) - NUM_SAMPLES) // 2
        audio = audio[start:start + NUM_SAMPLES]
    elif len(audio) < NUM_SAMPLES:
        audio = np.pad(audio, (0, NUM_SAMPLES - len(audio)))

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=0,
        fmax=SAMPLE_RATE / 2,
        power=2.0,
    )
    log_mel = librosa.power_to_db(mel, ref=1.0)
    return np.asarray(log_mel, dtype=np.float32)

feature_metadata = {}
start_time = time.time()

for split_name, manifest in SPLITS.items():
    cache_path = FEATURE_ROOT / f"{split_name}_features.npz"
    metadata_path = FEATURE_ROOT / f"{split_name}_metadata.csv"

    # Avoid accidental replacement of a previously generated cache.
    if cache_path.exists() or metadata_path.exists():
        raise FileExistsError(
            f"Cache already exists for {split_name}: {cache_path}. "
            "Inspect it before deciding whether to reuse or create a new version."
        )

    features = []
    metadata_rows = []

    for i, row in enumerate(manifest.itertuples(index=False), start=1):
        path = Path(row.recording_path)

        if not path.is_file():
            raise FileNotFoundError(f"Missing recording: {path}")

        feature = extract_logmel(path)

        if feature.shape[0] != N_MELS or not np.isfinite(feature).all():
            raise ValueError(f"Invalid feature for {path}")

        features.append(feature)
        metadata_rows.append({
            "source_dataset": row.source_dataset,
            "source_recording_id": row.source_recording_id,
            "patient_id": row.patient_id,
            "diagnosis": row.diagnosis,
            "recording_path": str(path),
        })

        if i % 50 == 0 or i == len(manifest):
            print(f"{split_name}: {i}/{len(manifest)} recordings processed")

    X = np.stack(features).astype(np.float32)
    metadata = pd.DataFrame(metadata_rows)

    np.savez_compressed(cache_path, X=X)
    metadata.to_csv(metadata_path, index=False)

    feature_metadata[split_name] = {
        "features_path": str(cache_path),
        "metadata_path": str(metadata_path),
        "shape": list(X.shape),
        "dtype": str(X.dtype),
    }

    print(f"\nSaved {split_name}: {X.shape} -> {cache_path}")

metadata_json_path = FEATURE_ROOT / "feature_extraction_config.json"
metadata_json_path.write_text(
    json.dumps({
        "sample_rate": SAMPLE_RATE,
        "duration_seconds": DURATION_SECONDS,
        "num_samples": NUM_SAMPLES,
        "n_fft": N_FFT,
        "hop_length": HOP_LENGTH,
        "n_mels": N_MELS,
        "window_policy": "one deterministic central 5-second crop per recording; pad shorter recordings",
        "feature_type": "log-Mel spectrogram in dB, ref=1.0",
        "splits": feature_metadata,
    }, indent=2),
    encoding="utf-8",
)

print("\n=== FEATURE EXTRACTION COMPLETE ===")
print("Feature directory:", FEATURE_ROOT)
print("Elapsed seconds:", round(time.time() - start_time, 1))
print("Config saved:", metadata_json_path)


train: 50/530 recordings processed
train: 100/530 recordings processed
train: 150/530 recordings processed
train: 200/530 recordings processed
train: 250/530 recordings processed
train: 300/530 recordings processed
train: 350/530 recordings processed
train: 400/530 recordings processed
train: 450/530 recordings processed
train: 500/530 recordings processed
train: 530/530 recordings processed

Saved train: (530, 64, 157) -> /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/features_logmel_v1/train_features.npz
validation: 50/102 recordings processed
validation: 100/102 recordings processed
validation: 102/102 recordings processed

Saved validation: (102, 64, 157) -> /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/features_logmel_v1/validation_features.npz
icbhi_test: 50/394 re

In [8]:

import json
import numpy as np
from pathlib import Path

FEATURE_ROOT = OUTPUT_ROOT / "features_logmel_v1"

# Load fresh feature caches.
X_train = np.load(
    FEATURE_ROOT / "train_features.npz"
)["X"].astype(np.float32)

X_val = np.load(
    FEATURE_ROOT / "validation_features.npz"
)["X"].astype(np.float32)

X_icbhi_test = np.load(
    FEATURE_ROOT / "icbhi_test_features.npz"
)["X"].astype(np.float32)

X_fraiwan_test = np.load(
    FEATURE_ROOT / "fraiwan_test_features.npz"
)["X"].astype(np.float32)

# Fit one global mean and standard deviation on training features only.
train_mean = float(X_train.mean())
train_std = float(X_train.std())

if not np.isfinite(train_mean) or not np.isfinite(train_std) or train_std <= 0:
    raise ValueError("Invalid training normalization statistics.")

def normalize_features(X):
    return ((X - train_mean) / train_std).astype(np.float32)

X_train_norm = normalize_features(X_train)
X_val_norm = normalize_features(X_val)
X_icbhi_test_norm = normalize_features(X_icbhi_test)
X_fraiwan_test_norm = normalize_features(X_fraiwan_test)

print("=== TRAINING-ONLY NORMALIZATION ===")
print("Training mean:", train_mean)
print("Training std:", train_std)
print("Normalized training mean:", float(X_train_norm.mean()))
print("Normalized training std:", float(X_train_norm.std()))

for name, X in [
    ("train", X_train_norm),
    ("validation", X_val_norm),
    ("icbhi_test", X_icbhi_test_norm),
    ("fraiwan_test", X_fraiwan_test_norm),
]:
    print(
        f"{name}: shape={X.shape}, "
        f"finite={bool(np.isfinite(X).all())}, "
        f"mean={X.mean():.4f}, std={X.std():.4f}"
    )

# Save as a separate normalized cache.
NORMALIZED_ROOT = OUTPUT_ROOT / "features_logmel_v1_normalized"
NORMALIZED_ROOT.mkdir(parents=True, exist_ok=True)

for name, X in [
    ("train", X_train_norm),
    ("validation", X_val_norm),
    ("icbhi_test", X_icbhi_test_norm),
    ("fraiwan_test", X_fraiwan_test_norm),
]:
    path = NORMALIZED_ROOT / f"{name}_features_normalized.npz"
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite existing cache: {path}")
    np.savez_compressed(path, X=X)

normalization_config = {
    "method": "global training-only standardization",
    "training_mean": train_mean,
    "training_std": train_std,
    "fit_split": "train",
    "feature_type": "log-Mel spectrogram",
}

norm_config_path = NORMALIZED_ROOT / "normalization_config.json"
if norm_config_path.exists():
    raise FileExistsError(f"Refusing to overwrite: {norm_config_path}")

norm_config_path.write_text(
    json.dumps(normalization_config, indent=2),
    encoding="utf-8",
)

print("\nNormalized feature cache saved to:", NORMALIZED_ROOT)
print("Validation and test data were not used to fit normalization.")


=== TRAINING-ONLY NORMALIZATION ===
Training mean: -39.280704498291016
Training std: 18.995901107788086
Normalized training mean: 2.3472239263355732e-08
Normalized training std: 1.0
train: shape=(530, 64, 157), finite=True, mean=0.0000, std=1.0000
validation: shape=(102, 64, 157), finite=True, mean=-0.0700, std=1.0102
icbhi_test: shape=(394, 64, 157), finite=True, mean=0.1156, std=0.9357
fraiwan_test: shape=(28, 64, 157), finite=True, mean=-0.8090, std=1.0414

Normalized feature cache saved to: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/features_logmel_v1_normalized
Validation and test data were not used to fit normalization.


In [10]:

import torch
import torch.nn as nn

# Select available accelerator
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")


class CompactFeatureBranch(nn.Module):
    """Small CNN branch for one time-frequency feature family."""

    def __init__(self, out_channels=24):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 12, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(12),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(12, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                out_channels, out_channels,
                kernel_size=3, padding=1, groups=1, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )

    def forward(self, x):
        return self.features(x.unsqueeze(1))


class CompactMultiFeatureCNN(nn.Module):
    def __init__(self, num_classes=8, dropout=0.30):
        super().__init__()

        self.logmel_branch = CompactFeatureBranch(out_channels=24)
        self.mfcc_branch = CompactFeatureBranch(out_channels=16)
        self.chroma_branch = CompactFeatureBranch(out_channels=16)

        fused_size = 24 + 16 + 16

        self.classifier = nn.Sequential(
            nn.Linear(fused_size, 48),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(48, num_classes),
        )

    def forward(self, logmel, mfcc, chroma):
        mel_features = self.logmel_branch(logmel)
        mfcc_features = self.mfcc_branch(mfcc)
        chroma_features = self.chroma_branch(chroma)

        fused = torch.cat(
            [mel_features, mfcc_features, chroma_features],
            dim=1,
        )
        return self.classifier(fused)


model = CompactMultiFeatureCNN(num_classes=len(CLASS_NAMES)).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

# Shape check with the planned feature dimensions.
dummy_logmel = torch.zeros(2, N_MELS, 157, device=DEVICE)
dummy_mfcc = torch.zeros(2, N_MFCC, 157, device=DEVICE)
dummy_chroma = torch.zeros(2, N_CHROMA, 157, device=DEVICE)

model.eval()
with torch.no_grad():
    output = model(dummy_logmel, dummy_mfcc, dummy_chroma)

print("=== COMPACT MULTI-FEATURE CNN ===")
print("Device:", DEVICE)
print("Log-Mel input:", tuple(dummy_logmel.shape))
print("MFCC input:", tuple(dummy_mfcc.shape))
print("Chroma input:", tuple(dummy_chroma.shape))
print("Output shape:", tuple(output.shape))
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Parameter reduction vs 100K:", f"{(1 - trainable_params / 100000) * 100:.1f}%")

assert output.shape == (2, len(CLASS_NAMES))
print("\nForward-pass check passed.")


=== COMPACT MULTI-FEATURE CNN ===
Device: mps
Log-Mel input: (2, 64, 157)
MFCC input: (2, 13, 157)
Chroma input: (2, 12, 157)
Output shape: (2, 8)
Total parameters: 19,588
Trainable parameters: 19,588
Parameter reduction vs 100K: 80.4%

Forward-pass check passed.


In [11]:

import time
import numpy as np
import pandas as pd
import librosa
from pathlib import Path

MULTIFEATURE_ROOT = OUTPUT_ROOT / "features_multifeature_v1"
MULTIFEATURE_ROOT.mkdir(parents=True, exist_ok=True)

MULTIFEATURE_SPLITS = {
    "train": c2_train_df,
    "validation": c2_val_df,
    "icbhi_test": icbhi_test_df,
    "fraiwan_test": fraiwan_test_df,
}

def extract_three_features(recording_path):
    audio, _ = librosa.load(
        str(recording_path),
        sr=SAMPLE_RATE,
        mono=True,
    )
    audio = np.asarray(audio, dtype=np.float32)

    if len(audio) > NUM_SAMPLES:
        start = (len(audio) - NUM_SAMPLES) // 2
        audio = audio[start:start + NUM_SAMPLES]
    elif len(audio) < NUM_SAMPLES:
        audio = np.pad(audio, (0, NUM_SAMPLES - len(audio)))

    mel_power = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=0,
        fmax=SAMPLE_RATE / 2,
        power=2.0,
    )
    logmel = librosa.power_to_db(mel_power, ref=1.0)

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=SAMPLE_RATE,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
    )

    chroma = librosa.feature.chroma_stft(
        y=audio,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )

    features = {
        "logmel": np.asarray(logmel, dtype=np.float32),
        "mfcc": np.asarray(mfcc, dtype=np.float32),
        "chroma": np.asarray(chroma, dtype=np.float32),
    }

    for name, array in features.items():
        if not np.isfinite(array).all():
            raise ValueError(f"Non-finite {name} features: {recording_path}")

    return features


start_time = time.time()

for split_name, manifest in MULTIFEATURE_SPLITS.items():
    output_path = MULTIFEATURE_ROOT / f"{split_name}_features.npz"
    metadata_path = MULTIFEATURE_ROOT / f"{split_name}_metadata.csv"

    if output_path.exists() or metadata_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing {split_name} cache: {output_path}"
        )

    feature_lists = {"logmel": [], "mfcc": [], "chroma": []}
    metadata_rows = []

    for i, row in enumerate(manifest.itertuples(index=False), start=1):
        path = Path(row.recording_path)
        if not path.is_file():
            raise FileNotFoundError(path)

        features = extract_three_features(path)

        for name in feature_lists:
            feature_lists[name].append(features[name])

        metadata_rows.append({
            "source_dataset": row.source_dataset,
            "source_recording_id": row.source_recording_id,
            "patient_id": row.patient_id,
            "diagnosis": row.diagnosis,
            "recording_path": str(path),
        })

        if i % 50 == 0 or i == len(manifest):
            print(f"{split_name}: {i}/{len(manifest)}")

    arrays = {
        name: np.stack(values).astype(np.float32)
        for name, values in feature_lists.items()
    }

    np.savez_compressed(output_path, **arrays)
    pd.DataFrame(metadata_rows).to_csv(metadata_path, index=False)

    print(f"\nSaved {split_name}:")
    for name, array in arrays.items():
        print(f"  {name}: {array.shape}")
    print("  File:", output_path)

print("\nElapsed seconds:", round(time.time() - start_time, 1))
print("Three-feature extraction complete.")


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/librosa/core/pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


train: 50/530
train: 100/530
train: 150/530
train: 200/530
train: 250/530
train: 300/530
train: 350/530
train: 400/530
train: 450/530
train: 500/530
train: 530/530

Saved train:
  logmel: (530, 64, 157)
  mfcc: (530, 13, 157)
  chroma: (530, 12, 157)
  File: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/features_multifeature_v1/train_features.npz


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/librosa/core/pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


validation: 50/102
validation: 100/102
validation: 102/102

Saved validation:
  logmel: (102, 64, 157)
  mfcc: (102, 13, 157)
  chroma: (102, 12, 157)
  File: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/features_multifeature_v1/validation_features.npz


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/librosa/core/pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


icbhi_test: 50/394
icbhi_test: 100/394
icbhi_test: 150/394
icbhi_test: 200/394
icbhi_test: 250/394
icbhi_test: 300/394
icbhi_test: 350/394
icbhi_test: 394/394

Saved icbhi_test:
  logmel: (394, 64, 157)
  mfcc: (394, 13, 157)
  chroma: (394, 12, 157)
  File: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/features_multifeature_v1/icbhi_test_features.npz
fraiwan_test: 28/28

Saved fraiwan_test:
  logmel: (28, 64, 157)
  mfcc: (28, 13, 157)
  chroma: (28, 12, 157)
  File: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/features_multifeature_v1/fraiwan_test_features.npz

Elapsed seconds: 19.2
Three-feature extraction complete.


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/librosa/core/pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


In [12]:

import json
import numpy as np
from pathlib import Path

MULTIFEATURE_ROOT = OUTPUT_ROOT / "features_multifeature_v1"
NORMALIZED_MULTI_ROOT = OUTPUT_ROOT / "features_multifeature_v1_normalized"
NORMALIZED_MULTI_ROOT.mkdir(parents=True, exist_ok=True)

FEATURE_NAMES = ["logmel", "mfcc", "chroma"]
SPLIT_NAMES = ["train", "validation", "icbhi_test", "fraiwan_test"]

raw_features = {}

for split in SPLIT_NAMES:
    path = MULTIFEATURE_ROOT / f"{split}_features.npz"
    with np.load(path) as data:
        raw_features[split] = {
            name: data[name].astype(np.float32)
            for name in FEATURE_NAMES
        }

# Fit separate global statistics for each feature family using train only.
normalization_stats = {}

for name in FEATURE_NAMES:
    train_array = raw_features["train"][name]

    mean = float(train_array.mean())
    std = float(train_array.std())

    if not np.isfinite(mean) or not np.isfinite(std) or std <= 0:
        raise ValueError(f"Invalid normalization statistics for {name}")

    normalization_stats[name] = {"mean": mean, "std": std}

print("=== TRAINING-ONLY NORMALIZATION STATISTICS ===")
for name in FEATURE_NAMES:
    print(
        f"{name}: mean={normalization_stats[name]['mean']:.6f}, "
        f"std={normalization_stats[name]['std']:.6f}"
    )

# Normalize all splits with the corresponding training-only statistics.
for split in SPLIT_NAMES:
    normalized_arrays = {}

    for name in FEATURE_NAMES:
        stats = normalization_stats[name]
        normalized = (
            raw_features[split][name] - stats["mean"]
        ) / stats["std"]

        normalized = normalized.astype(np.float32)

        if not np.isfinite(normalized).all():
            raise ValueError(f"Non-finite normalized values: {split}/{name}")

        normalized_arrays[name] = normalized

    output_path = NORMALIZED_MULTI_ROOT / f"{split}_features_normalized.npz"

    if output_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing normalized cache: {output_path}"
        )

    np.savez_compressed(output_path, **normalized_arrays)

    print(f"\n{split}:")
    for name, array in normalized_arrays.items():
        print(
            f"  {name}: shape={array.shape}, "
            f"mean={array.mean():.4f}, std={array.std():.4f}, "
            f"finite={bool(np.isfinite(array).all())}"
        )

config_path = NORMALIZED_MULTI_ROOT / "normalization_config.json"
if config_path.exists():
    raise FileExistsError(f"Refusing to overwrite: {config_path}")

config_path.write_text(
    json.dumps({
        "method": "global standardization per feature family",
        "fit_split": "train",
        "statistics": normalization_stats,
        "feature_shapes": {
            "logmel": [N_MELS, 157],
            "mfcc": [N_MFCC, 157],
            "chroma": [N_CHROMA, 157],
        },
    }, indent=2),
    encoding="utf-8",
)

print("\nNormalized feature cache:", NORMALIZED_MULTI_ROOT)
print("Normalization configuration saved.")



=== TRAINING-ONLY NORMALIZATION STATISTICS ===
logmel: mean=-39.280704, std=18.995901
mfcc: mean=-6.041240, std=96.432480
chroma: mean=0.708381, std=0.223160

train:
  logmel: shape=(530, 64, 157), mean=0.0000, std=1.0000, finite=True
  mfcc: shape=(530, 13, 157), mean=-0.0000, std=1.0000, finite=True
  chroma: shape=(530, 12, 157), mean=-0.0000, std=1.0000, finite=True

validation:
  logmel: shape=(102, 64, 157), mean=-0.0700, std=1.0102, finite=True
  mfcc: shape=(102, 13, 157), mean=-0.0027, std=1.0295, finite=True
  chroma: shape=(102, 12, 157), mean=0.0517, std=0.9759, finite=True

icbhi_test:
  logmel: shape=(394, 64, 157), mean=0.1156, std=0.9357, finite=True
  mfcc: shape=(394, 13, 157), mean=0.0056, std=0.9425, finite=True
  chroma: shape=(394, 12, 157), mean=-0.0306, std=1.0189, finite=True

fraiwan_test:
  logmel: shape=(28, 64, 157), mean=-0.8090, std=1.0414, finite=True
  mfcc: shape=(28, 13, 157), mean=-0.0613, std=1.3299, finite=True
  chroma: shape=(28, 12, 157), mean=0

In [13]:

import numpy as np
import pandas as pd
import torch
from sklearn.utils.class_weight import compute_class_weight

NORMALIZED_MULTI_ROOT = OUTPUT_ROOT / "features_multifeature_v1_normalized"
MULTIFEATURE_ROOT = OUTPUT_ROOT / "features_multifeature_v1"

def load_multifeature_split(split_name):
    path = NORMALIZED_MULTI_ROOT / f"{split_name}_features_normalized.npz"
    with np.load(path) as data:
        return {
            name: data[name].astype(np.float32)
            for name in ("logmel", "mfcc", "chroma")
        }

X_train_multi = load_multifeature_split("train")
X_val_multi = load_multifeature_split("validation")
X_icbhi_test_multi = load_multifeature_split("icbhi_test")
X_fraiwan_test_multi = load_multifeature_split("fraiwan_test")

train_meta = pd.read_csv(MULTIFEATURE_ROOT / "train_metadata.csv")
val_meta = pd.read_csv(MULTIFEATURE_ROOT / "validation_metadata.csv")
icbhi_test_meta = pd.read_csv(MULTIFEATURE_ROOT / "icbhi_test_metadata.csv")
fraiwan_test_meta = pd.read_csv(MULTIFEATURE_ROOT / "fraiwan_test_metadata.csv")

def encode_labels(metadata):
    if metadata["diagnosis"].isna().any():
        raise ValueError("Missing diagnosis labels found.")

    unknown = set(metadata["diagnosis"]) - set(CLASS_TO_IDX)
    if unknown:
        raise ValueError(f"Unexpected diagnosis labels: {unknown}")

    return metadata["diagnosis"].map(CLASS_TO_IDX).to_numpy(dtype=np.int64)

y_train = encode_labels(train_meta)
y_val = encode_labels(val_meta)
y_icbhi_test = encode_labels(icbhi_test_meta)
y_fraiwan_test = encode_labels(fraiwan_test_meta)

for name, features, labels, metadata in [
    ("train", X_train_multi, y_train, train_meta),
    ("validation", X_val_multi, y_val, val_meta),
    ("ICBHI test", X_icbhi_test_multi, y_icbhi_test, icbhi_test_meta),
    ("Fraiwan test", X_fraiwan_test_multi, y_fraiwan_test, fraiwan_test_meta),
]:
    lengths = {len(array) for array in features.values()}
    assert lengths == {len(labels), len(metadata)}, (
        f"Feature/label/metadata mismatch in {name}"
    )
    assert all(np.isfinite(array).all() for array in features.values())
    print(f"{name}: {len(labels)} recordings aligned")

# Calculate class weights from training labels only.
present_classes = np.unique(y_train)
raw_weights = compute_class_weight(
    class_weight="balanced",
    classes=present_classes,
    y=y_train,
)

class_weights = np.zeros(len(CLASS_NAMES), dtype=np.float32)
class_weights[present_classes] = raw_weights.astype(np.float32)

# Cap extreme weights, then normalize weights over observed training classes.
class_weights = np.minimum(class_weights, 4.0)
class_weights[present_classes] /= class_weights[present_classes].mean()

train_support = np.bincount(y_train, minlength=len(CLASS_NAMES))

print("\n=== TRAINING CLASS SUPPORT AND WEIGHTS ===")
display(pd.DataFrame({
    "class": CLASS_NAMES,
    "training_recordings": train_support,
    "class_weight": class_weights,
}))

print("\nClasses absent from this training split:")
print([CLASS_NAMES[i] for i in range(len(CLASS_NAMES)) if train_support[i] == 0])

# Convert feature arrays to CPU tensors; move batches to MPS during training.
X_train_tensors = {
    name: torch.from_numpy(array)
    for name, array in X_train_multi.items()
}
X_val_tensors = {
    name: torch.from_numpy(array)
    for name, array in X_val_multi.items()
}
X_icbhi_test_tensors = {
    name: torch.from_numpy(array)
    for name, array in X_icbhi_test_multi.items()
}
X_fraiwan_test_tensors = {
    name: torch.from_numpy(array)
    for name, array in X_fraiwan_test_multi.items()
}

y_train_tensor = torch.from_numpy(y_train)
y_val_tensor = torch.from_numpy(y_val)
y_icbhi_test_tensor = torch.from_numpy(y_icbhi_test)
y_fraiwan_test_tensor = torch.from_numpy(y_fraiwan_test)
class_weights_tensor = torch.from_numpy(class_weights)

print("\nTensor preparation complete.")


train: 530 recordings aligned
validation: 102 recordings aligned
ICBHI test: 394 recordings aligned
Fraiwan test: 28 recordings aligned

=== TRAINING CLASS SUPPORT AND WEIGHTS ===


,class,training_recordings,class_weight
0,Asthma,65,0.425356
1,Bronchiectasis,14,1.460657
2,Bronchiolitis,7,1.460657
3,COPD,375,0.073728
4,Healthy,11,1.460657
5,LRTI,0,0.000000
6,Pneumonia,42,0.658289
7,URTI,16,1.460657



Classes absent from this training split:
['LRTI']

Tensor preparation complete.


/var/folders/xp/3qf723ln63q9ncf3lc_ygq1c0000gp/T/ipykernel_31896/3351056653.py:100: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  y_train_tensor = torch.from_numpy(y_train)


In [14]:

import pandas as pd

source_train = pd.read_csv(BASE_DIR / "experiment_b_train_manifest.csv").copy()

source_train["group_id"] = (
    source_train["source_dataset"].astype(str)
    + "::"
    + source_train["patient_id"].astype(str)
)

print("=== RARE-CLASS RECORDING SUPPORT ===")
rare_classes = ["LRTI", "Bronchiolitis", "Healthy", "Bronchiectasis", "URTI"]

for class_name in rare_classes:
    subset = source_train[source_train["diagnosis"] == class_name]
    print(f"\n{class_name}: {len(subset)} recordings")
    if len(subset):
        patient_summary = (
            subset.groupby(["source_dataset", "patient_id"])
            .size()
            .reset_index(name="recordings")
            .sort_values("recordings", ascending=False)
        )
        display(patient_summary)

print("\n=== LRTI RECORDING DETAILS ===")
display(
    source_train.loc[
        source_train["diagnosis"] == "LRTI",
        ["source_dataset", "source_recording_id", "patient_id", "recording_path"]
    ]
)

print("\n=== CURRENT C2 SPLIT SUPPORT ===")
for split_name, df in [
    ("train", c2_train_df),
    ("validation", c2_val_df),
]:
    print(f"\n{split_name}:")
    display(
        df.groupby(["source_dataset", "diagnosis"])
        .size()
        .unstack(fill_value=0)
    )


=== RARE-CLASS RECORDING SUPPORT ===

LRTI: 2 recordings


,source_dataset,patient_id,recordings
0,ICBHI,ICBHI_108,1
1,ICBHI,ICBHI_115,1



Bronchiolitis: 7 recordings


,source_dataset,patient_id,recordings
2,ICBHI,ICBHI_206,3
0,ICBHI,ICBHI_161,2
1,ICBHI,ICBHI_167,2



Healthy: 18 recordings


...

,source_dataset,source_recording_id,patient_id,recording_path
32,ICBHI,108_1b1_Al_sc_Meditron,ICBHI_108,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
50,ICBHI,115_1b1_Ar_sc_Meditron,ICBHI_115,/Users/vs/Sem 5/Respiratory-Disease-Classifica...



=== CURRENT C2 SPLIT SUPPORT ===

train:


diagnosis,Asthma,Bronchiectasis,Bronchiolitis,COPD,Healthy,Pneumonia,URTI
source_dataset,,,,,,,
Fraiwan,65,0,0,15,0,6,0
ICBHI,0,14,7,360,11,36,16



validation:


diagnosis,Asthma,COPD,Healthy,LRTI,Pneumonia
source_dataset,,,,,
Fraiwan,9,6,0,0,6
ICBHI,1,71,7,2,0


In [15]:

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

source_train = pd.read_csv(BASE_DIR / "experiment_b_train_manifest.csv").copy()

source_train["group_id"] = (
    source_train["source_dataset"].astype(str)
    + "::"
    + source_train["patient_id"].astype(str)
)

# Identify the two distinct LRTI patient groups.
lrti_groups = sorted(
    source_train.loc[source_train["diagnosis"] == "LRTI", "group_id"].unique()
)

if len(lrti_groups) != 2:
    raise ValueError(
        f"Expected exactly two LRTI patient groups, found {lrti_groups}"
    )

print("LRTI patient groups:", lrti_groups)

# Try deterministic grouped splits until one LRTI patient is in each split.
# This keeps every patient wholly in either training or validation.
chosen = None

for seed in range(42, 1042):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=seed,
    )
    train_idx, val_idx = next(
        splitter.split(source_train, groups=source_train["group_id"])
    )

    train_groups = set(source_train.iloc[train_idx]["group_id"])
    val_groups = set(source_train.iloc[val_idx]["group_id"])

    if (
        len(set(lrti_groups) & train_groups) == 1
        and len(set(lrti_groups) & val_groups) == 1
    ):
        chosen = (seed, train_idx, val_idx)
        break

if chosen is None:
    raise RuntimeError("Could not find a suitable grouped split.")

chosen_seed, train_idx, val_idx = chosen

c2_train_df = (
    source_train.iloc[train_idx]
    .drop(columns=["group_id"])
    .reset_index(drop=True)
)
c2_val_df = (
    source_train.iloc[val_idx]
    .drop(columns=["group_id"])
    .reset_index(drop=True)
)

train_groups = set(
    source_train.iloc[train_idx]["group_id"]
)
val_groups = set(
    source_train.iloc[val_idx]["group_id"]
)

assert train_groups.isdisjoint(val_groups)
assert len(set(lrti_groups) & train_groups) == 1
assert len(set(lrti_groups) & val_groups) == 1

print("\n=== CORRECTED C2 SPLIT ===")
print("Selected seed:", chosen_seed)
print("Training recordings:", len(c2_train_df))
print("Validation recordings:", len(c2_val_df))
print("Training patients:", c2_train_df["patient_id"].nunique())
print("Validation patients:", c2_val_df["patient_id"].nunique())
print("Patient overlap:", len(train_groups & val_groups))

print("\nTraining class support:")
display(c2_train_df["diagnosis"].value_counts().reindex(CLASS_NAMES, fill_value=0))

print("\nValidation class support:")
display(c2_val_df["diagnosis"].value_counts().reindex(CLASS_NAMES, fill_value=0))

print("\nLRTI assignment:")
display(
    pd.concat([
        c2_train_df[c2_train_df["diagnosis"] == "LRTI"].assign(split="train"),
        c2_val_df[c2_val_df["diagnosis"] == "LRTI"].assign(split="validation"),
    ])[["split", "source_dataset", "patient_id", "source_recording_id"]]
)


LRTI patient groups: ['ICBHI::ICBHI_108', 'ICBHI::ICBHI_115']

=== CORRECTED C2 SPLIT ===
Selected seed: 43
Training recordings: 501
Validation recordings: 131
Training patients: 90
Validation patients: 23
Patient overlap: 0

Training class support:


diagnosis
Asthma             48
Bronchiectasis     13
Bronchiolitis       7
COPD              366
Healthy            12
LRTI                1
Pneumonia          41
URTI               13
Name: count, dtype: int64


Validation class support:


diagnosis
Asthma            27
Bronchiectasis     1
Bronchiolitis      0
COPD              86
Healthy            6
LRTI               1
Pneumonia          7
URTI               3
Name: count, dtype: int64


LRTI assignment:


,split,source_dataset,patient_id,source_recording_id
31,train,ICBHI,ICBHI_108,108_1b1_Al_sc_Meditron
1,validation,ICBHI,ICBHI_115,115_1b1_Ar_sc_Meditron


In [16]:

# Save corrected C2 train and validation manifests.
# Original Experiment B manifests and C2 test manifests remain unchanged.

c2_train_path = OUTPUT_ROOT / "c2_train_manifest.csv"
c2_val_path = OUTPUT_ROOT / "c2_validation_manifest.csv"

if c2_train_path.exists():
    print("Existing C2 training manifest will be backed up.")
    backup_path = OUTPUT_ROOT / "c2_train_manifest_before_lrti_fix.csv"
    if not backup_path.exists():
        pd.read_csv(c2_train_path).to_csv(backup_path, index=False)

if c2_val_path.exists():
    print("Existing C2 validation manifest will be backed up.")
    backup_path = OUTPUT_ROOT / "c2_validation_manifest_before_lrti_fix.csv"
    if not backup_path.exists():
        pd.read_csv(c2_val_path).to_csv(backup_path, index=False)

c2_train_df.to_csv(c2_train_path, index=False)
c2_val_df.to_csv(c2_val_path, index=False)

# Verify the saved manifests.
saved_train = pd.read_csv(c2_train_path)
saved_val = pd.read_csv(c2_val_path)

assert len(saved_train) == 501
assert len(saved_val) == 131
assert not (
    set(saved_train["patient_id"]) & set(saved_val["patient_id"])
)
assert (saved_train["diagnosis"] == "LRTI").sum() == 1
assert (saved_val["diagnosis"] == "LRTI").sum() == 1

print("\n=== CORRECTED MANIFESTS SAVED ===")
print("Training:", len(saved_train), "|", c2_train_path)
print("Validation:", len(saved_val), "|", c2_val_path)
print("LRTI training recordings:", int((saved_train["diagnosis"] == "LRTI").sum()))
print("LRTI validation recordings:", int((saved_val["diagnosis"] == "LRTI").sum()))
print("Patient overlap: 0")
print("Existing test manifests were not modified.")


Existing C2 training manifest will be backed up.
Existing C2 validation manifest will be backed up.

=== CORRECTED MANIFESTS SAVED ===
Training: 501 | /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/c2_train_manifest.csv
Validation: 131 | /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/c2_validation_manifest.csv
LRTI training recordings: 1
LRTI validation recordings: 1
Patient overlap: 0
Existing test manifests were not modified.


In [17]:

import time
import numpy as np
import pandas as pd
import librosa
from pathlib import Path

MULTIFEATURE_V2_ROOT = OUTPUT_ROOT / "features_multifeature_v2"
MULTIFEATURE_V2_ROOT.mkdir(parents=True, exist_ok=True)

CURRENT_SPLITS = {
    "train": c2_train_df,
    "validation": c2_val_df,
    "icbhi_test": icbhi_test_df,
    "fraiwan_test": fraiwan_test_df,
}

def extract_three_features_v2(recording_path):
    audio, _ = librosa.load(
        str(recording_path),
        sr=SAMPLE_RATE,
        mono=True,
    )
    audio = np.asarray(audio, dtype=np.float32)

    if len(audio) > NUM_SAMPLES:
        start = (len(audio) - NUM_SAMPLES) // 2
        audio = audio[start:start + NUM_SAMPLES]
    elif len(audio) < NUM_SAMPLES:
        audio = np.pad(audio, (0, NUM_SAMPLES - len(audio)))

    logmel = librosa.power_to_db(
        librosa.feature.melspectrogram(
            y=audio,
            sr=SAMPLE_RATE,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            fmin=0,
            fmax=SAMPLE_RATE / 2,
            power=2.0,
        ),
        ref=1.0,
    )

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=SAMPLE_RATE,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
    )

    chroma = librosa.feature.chroma_stft(
        y=audio,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )

    result = {
        "logmel": np.asarray(logmel, dtype=np.float32),
        "mfcc": np.asarray(mfcc, dtype=np.float32),
        "chroma": np.asarray(chroma, dtype=np.float32),
    }

    for name, array in result.items():
        if not np.isfinite(array).all():
            raise ValueError(f"Non-finite {name} features: {recording_path}")

    return result


start_time = time.time()

for split_name, manifest in CURRENT_SPLITS.items():
    feature_path = MULTIFEATURE_V2_ROOT / f"{split_name}_features.npz"
    metadata_path = MULTIFEATURE_V2_ROOT / f"{split_name}_metadata.csv"

    if feature_path.exists() or metadata_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing v2 cache: {feature_path}"
        )

    feature_lists = {"logmel": [], "mfcc": [], "chroma": []}
    metadata_rows = []

    for i, row in enumerate(manifest.itertuples(index=False), start=1):
        path = Path(row.recording_path)
        if not path.is_file():
            raise FileNotFoundError(path)

        features = extract_three_features_v2(path)

        for name in feature_lists:
            feature_lists[name].append(features[name])

        metadata_rows.append({
            "source_dataset": row.source_dataset,
            "source_recording_id": row.source_recording_id,
            "patient_id": row.patient_id,
            "diagnosis": row.diagnosis,
            "recording_path": str(path),
        })

        if i % 50 == 0 or i == len(manifest):
            print(f"{split_name}: {i}/{len(manifest)}")

    arrays = {
        name: np.stack(values).astype(np.float32)
        for name, values in feature_lists.items()
    }

    np.savez_compressed(feature_path, **arrays)
    pd.DataFrame(metadata_rows).to_csv(metadata_path, index=False)

    print(f"\nSaved {split_name}:")
    for name, array in arrays.items():
        print(f"  {name}: {array.shape}")
    print("  File:", feature_path)

print("\nElapsed seconds:", round(time.time() - start_time, 1))
print("Corrected three-feature cache created.")


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/librosa/core/pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


train: 50/501
train: 100/501
train: 150/501
train: 200/501
train: 250/501
train: 300/501
train: 350/501
train: 400/501
train: 450/501
train: 500/501
train: 501/501

Saved train:
  logmel: (501, 64, 157)
  mfcc: (501, 13, 157)
  chroma: (501, 12, 157)
  File: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/features_multifeature_v2/train_features.npz


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/librosa/core/pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


validation: 50/131
validation: 100/131
validation: 131/131

Saved validation:
  logmel: (131, 64, 157)
  mfcc: (131, 13, 157)
  chroma: (131, 12, 157)
  File: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/features_multifeature_v2/validation_features.npz


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/librosa/core/pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


icbhi_test: 50/394
icbhi_test: 100/394
icbhi_test: 150/394
icbhi_test: 200/394
icbhi_test: 250/394
icbhi_test: 300/394
icbhi_test: 350/394
icbhi_test: 394/394

Saved icbhi_test:
  logmel: (394, 64, 157)
  mfcc: (394, 13, 157)
  chroma: (394, 12, 157)
  File: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/features_multifeature_v2/icbhi_test_features.npz
fraiwan_test: 28/28

Saved fraiwan_test:
  logmel: (28, 64, 157)
  mfcc: (28, 13, 157)
  chroma: (28, 12, 157)
  File: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/features_multifeature_v2/fraiwan_test_features.npz

Elapsed seconds: 18.7
Corrected three-feature cache created.


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/librosa/core/pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


In [18]:

import json
import numpy as np
from pathlib import Path

MULTIFEATURE_V2_ROOT = OUTPUT_ROOT / "features_multifeature_v2"
NORMALIZED_V2_ROOT = OUTPUT_ROOT / "features_multifeature_v2_normalized"
NORMALIZED_V2_ROOT.mkdir(parents=True, exist_ok=True)

FEATURE_NAMES = ["logmel", "mfcc", "chroma"]
SPLIT_NAMES = ["train", "validation", "icbhi_test", "fraiwan_test"]

raw_features_v2 = {}

for split in SPLIT_NAMES:
    path = MULTIFEATURE_V2_ROOT / f"{split}_features.npz"
    with np.load(path) as data:
        raw_features_v2[split] = {
            name: data[name].astype(np.float32)
            for name in FEATURE_NAMES
        }

# Fit mean/std from corrected training data only.
normalization_stats_v2 = {}

for name in FEATURE_NAMES:
    arr = raw_features_v2["train"][name]
    mean = float(arr.mean())
    std = float(arr.std())

    if not np.isfinite(mean) or not np.isfinite(std) or std <= 0:
        raise ValueError(f"Invalid normalization statistics: {name}")

    normalization_stats_v2[name] = {"mean": mean, "std": std}

print("=== TRAINING-ONLY STATISTICS ===")
for name, stats in normalization_stats_v2.items():
    print(f"{name}: mean={stats['mean']:.6f}, std={stats['std']:.6f}")

for split in SPLIT_NAMES:
    normalized = {}

    for name in FEATURE_NAMES:
        stats = normalization_stats_v2[name]
        arr = (
            raw_features_v2[split][name] - stats["mean"]
        ) / stats["std"]
        arr = arr.astype(np.float32)

        if not np.isfinite(arr).all():
            raise ValueError(f"Non-finite values: {split}/{name}")

        normalized[name] = arr

    out_path = NORMALIZED_V2_ROOT / f"{split}_features_normalized.npz"

    if out_path.exists():
        raise FileExistsError(f"Refusing to overwrite: {out_path}")

    np.savez_compressed(out_path, **normalized)

    print(f"\n{split}:")
    for name, arr in normalized.items():
        print(
            f"  {name}: shape={arr.shape}, "
            f"mean={arr.mean():.4f}, std={arr.std():.4f}, "
            f"finite={np.isfinite(arr).all()}"
        )

config_path = NORMALIZED_V2_ROOT / "normalization_config.json"
if config_path.exists():
    raise FileExistsError(f"Refusing to overwrite: {config_path}")

config_path.write_text(
    json.dumps({
        "method": "global standardization per feature family",
        "fit_split": "train",
        "statistics": normalization_stats_v2,
    }, indent=2),
    encoding="utf-8",
)

print("\nNormalized v2 cache saved:", NORMALIZED_V2_ROOT)


=== TRAINING-ONLY STATISTICS ===
logmel: mean=-38.733738, std=18.971832
mfcc: mean=-5.886697, std=95.320946
chroma: mean=0.707414, std=0.223571

train:
  logmel: shape=(501, 64, 157), mean=-0.0000, std=1.0000, finite=True
  mfcc: shape=(501, 13, 157), mean=-0.0000, std=1.0000, finite=True
  chroma: shape=(501, 12, 157), mean=-0.0000, std=1.0000, finite=True

validation:
  logmel: shape=(131, 64, 157), mean=-0.1937, std=1.0008, finite=True
  mfcc: shape=(131, 13, 157), mean=-0.0099, std=1.0774, finite=True
  chroma: shape=(131, 12, 157), mean=0.0611, std=0.9717, finite=True

icbhi_test:
  logmel: shape=(394, 64, 157), mean=0.0869, std=0.9368, finite=True
  mfcc: shape=(394, 13, 157), mean=0.0040, std=0.9535, finite=True
  chroma: shape=(394, 12, 157), mean=-0.0262, std=1.0170, finite=True

fraiwan_test:
  logmel: shape=(28, 64, 157), mean=-0.8388, std=1.0428, finite=True
  mfcc: shape=(28, 13, 157), mean=-0.0637, std=1.3454, finite=True
  chroma: shape=(28, 12, 157), mean=0.0423, std=1.

In [19]:

import numpy as np
import pandas as pd
import torch

from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import TensorDataset, DataLoader

CLASS_NAMES = [
    "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
    "Healthy", "LRTI", "Pneumonia", "URTI"
]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

NORMALIZED_V2_ROOT = OUTPUT_ROOT / "features_multifeature_v2_normalized"

split_frames = {
    "train": c2_train_df.reset_index(drop=True).copy(),
    "validation": c2_val_df.reset_index(drop=True).copy(),
    "icbhi_test": icbhi_test_df.reset_index(drop=True).copy(),
    "fraiwan_test": fraiwan_test_df.reset_index(drop=True).copy(),
}

loaded_features = {}

for split, frame in split_frames.items():
    feature_path = NORMALIZED_V2_ROOT / f"{split}_features_normalized.npz"

    with np.load(feature_path) as data:
        features = {
            name: data[name].astype(np.float32)
            for name in ["logmel", "mfcc", "chroma"]
        }

    if any(len(arr) != len(frame) for arr in features.values()):
        raise ValueError(f"Feature/manifest row count mismatch: {split}")

    labels = frame["diagnosis"].map(CLASS_TO_IDX)

    if labels.isna().any():
        raise ValueError(
            f"Unknown diagnosis in {split}: "
            f"{frame.loc[labels.isna(), 'diagnosis'].unique()}"
        )

    y = labels.to_numpy(dtype=np.int64)

    for name, arr in features.items():
        if not np.isfinite(arr).all():
            raise ValueError(f"Non-finite {name} values in {split}")

    loaded_features[split] = {
        **features,
        "y": y,
        "metadata": frame,
    }

    print(
        f"{split}: {len(y)} recordings | "
        f"labels aligned | "
        f"Log-Mel {features['logmel'].shape} | "
        f"MFCC {features['mfcc'].shape} | "
        f"Chroma {features['chroma'].shape}"
    )

# Training-only class weights.
y_train = loaded_features["train"]["y"]
present_classes = np.unique(y_train)

raw_weights = compute_class_weight(
    class_weight="balanced",
    classes=present_classes,
    y=y_train,
)

class_weights_np = np.zeros(len(CLASS_NAMES), dtype=np.float32)
class_weights_np[present_classes] = raw_weights

# Cap extreme weights caused by very rare classes.
class_weights_np = np.minimum(class_weights_np, 4.0)
class_weights_np[present_classes] /= class_weights_np[present_classes].mean()

class_weights = torch.tensor(class_weights_np, dtype=torch.float32)

print("\n=== TRAINING CLASS SUPPORT ===")
for i, name in enumerate(CLASS_NAMES):
    count = int((y_train == i).sum())
    print(f"{name:18s}: {count:3d} | weight={class_weights_np[i]:.3f}")

print("\n=== VALIDATION CLASS SUPPORT ===")
y_val = loaded_features["validation"]["y"]
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:18s}: {int((y_val == i).sum()):3d}")

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("\nTraining device:", DEVICE)


train: 501 recordings | labels aligned | Log-Mel (501, 64, 157) | MFCC (501, 13, 157) | Chroma (501, 12, 157)
validation: 131 recordings | labels aligned | Log-Mel (131, 64, 157) | MFCC (131, 13, 157) | Chroma (131, 12, 157)
icbhi_test: 394 recordings | labels aligned | Log-Mel (394, 64, 157) | MFCC (394, 13, 157) | Chroma (394, 12, 157)
fraiwan_test: 28 recordings | labels aligned | Log-Mel (28, 64, 157) | MFCC (28, 13, 157) | Chroma (28, 12, 157)

=== TRAINING CLASS SUPPORT ===
Asthma            :  48 | weight=0.454
Bronchiectasis    :  13 | weight=1.391
Bronchiolitis     :   7 | weight=1.391
COPD              : 366 | weight=0.060
Healthy           :  12 | weight=1.391
LRTI              :   1 | weight=1.391
Pneumonia         :  41 | weight=0.531
URTI              :  13 | weight=1.391

=== VALIDATION CLASS SUPPORT ===
Asthma            :  27
Bronchiectasis    :   1
Bronchiolitis     :   0
COPD              :  86
Healthy           :   6
LRTI              :   1
Pneumonia         :   7
U

In [20]:

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Compact CNN: separate branches for Log-Mel, MFCC, and chroma.
class CompactFeatureBranch(nn.Module):
    def __init__(self, out_channels=24):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 12, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(12),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(12, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                out_channels, out_channels,
                kernel_size=3, padding=1, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )

    def forward(self, x):
        return self.features(x.unsqueeze(1))


class CompactMultiFeatureCNN(nn.Module):
    def __init__(self, num_classes=8, dropout=0.30):
        super().__init__()

        self.logmel_branch = CompactFeatureBranch(24)
        self.mfcc_branch = CompactFeatureBranch(16)
        self.chroma_branch = CompactFeatureBranch(16)

        self.classifier = nn.Sequential(
            nn.Linear(24 + 16 + 16, 48),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(48, num_classes),
        )

    def forward(self, logmel, mfcc, chroma):
        mel_features = self.logmel_branch(logmel)
        mfcc_features = self.mfcc_branch(mfcc)
        chroma_features = self.chroma_branch(chroma)

        fused = torch.cat(
            [mel_features, mfcc_features, chroma_features],
            dim=1,
        )
        return self.classifier(fused)


def make_dataset(split):
    data = loaded_features[split]

    # Copy arrays before tensor conversion to avoid non-writable-array warnings.
    return TensorDataset(
        torch.from_numpy(data["logmel"].copy()),
        torch.from_numpy(data["mfcc"].copy()),
        torch.from_numpy(data["chroma"].copy()),
        torch.from_numpy(data["y"].copy()).long(),
    )


BATCH_SIZE = 32
SEED = 42

torch.manual_seed(SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

train_loader = DataLoader(
    make_dataset("train"),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)
val_loader = DataLoader(
    make_dataset("validation"),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

model = CompactMultiFeatureCNN(num_classes=len(CLASS_NAMES)).to(DEVICE)

trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

# Smoke test using one batch.
batch = next(iter(train_loader))
with torch.no_grad():
    logits = model(
        batch[0][:2].to(DEVICE),
        batch[1][:2].to(DEVICE),
        batch[2][:2].to(DEVICE),
    )

print("Device:", DEVICE)
print("Trainable parameters:", f"{trainable_params:,}")
print("Model output shape:", tuple(logits.shape))
print("Expected output shape:", (2, len(CLASS_NAMES)))
print("Training batches per epoch:", len(train_loader))
print("Validation batches:", len(val_loader))

assert trainable_params == 19588, (
    f"Parameter count differs from expected: {trainable_params}"
)
assert logits.shape == (2, len(CLASS_NAMES))
print("\nModel and data loaders are ready.")


Device: mps
Trainable parameters: 19,588
Model output shape: (2, 8)
Expected output shape: (2, 8)
Training batches per epoch: 16
Validation batches: 5

Model and data loaders are ready.


In [21]:

import json
import copy
import random
import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import f1_score

SEED = 42
MAX_EPOCHS = 100
PATIENCE = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

# Class weights were computed from training labels only in Cell 17.
criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

# Macro-F1 is calculated only over classes represented in validation.
y_val_reference = loaded_features["validation"]["y"]
validation_classes = np.unique(y_val_reference)

checkpoint_path = OUTPUT_ROOT / "c2_compact_multifeature_best.pt"
history_path = OUTPUT_ROOT / "c2_compact_multifeature_training_history.json"

if checkpoint_path.exists() or history_path.exists():
    raise FileExistsError(
        "Training output already exists. Choose a new versioned filename "
        "before running again; existing results will not be overwritten."
    )

history = []
best_val_macro_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0
best_state = None

for epoch in range(1, MAX_EPOCHS + 1):
    # ---------------- Training ----------------
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    for logmel, mfcc, chroma, labels in train_loader:
        logmel = logmel.to(DEVICE)
        mfcc = mfcc.to(DEVICE)
        chroma = chroma.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        logits = model(logmel, mfcc, chroma)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_total += batch_size

    train_loss = running_loss / train_total
    train_accuracy = train_correct / train_total

    # ---------------- Validation ----------------
    model.eval()
    val_loss_sum = 0.0
    val_total = 0
    val_predictions = []
    val_targets = []

    with torch.no_grad():
        for logmel, mfcc, chroma, labels in val_loader:
            logmel = logmel.to(DEVICE)
            mfcc = mfcc.to(DEVICE)
            chroma = chroma.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(logmel, mfcc, chroma)
            loss = criterion(logits, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_total += labels.size(0)

            val_predictions.extend(logits.argmax(dim=1).cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    val_loss = val_loss_sum / val_total
    val_predictions = np.asarray(val_predictions)
    val_targets = np.asarray(val_targets)

    val_accuracy = float(np.mean(val_predictions == val_targets))
    val_macro_f1 = f1_score(
        val_targets,
        val_predictions,
        labels=validation_classes,
        average="macro",
        zero_division=0,
    )

    row = {
        "epoch": epoch,
        "train_loss": float(train_loss),
        "train_accuracy": float(train_accuracy),
        "validation_loss": float(val_loss),
        "validation_accuracy": float(val_accuracy),
        "validation_macro_f1_supported_classes": float(val_macro_f1),
    }
    history.append(row)

    improved = val_macro_f1 > best_val_macro_f1 + 1e-6

    if improved:
        best_val_macro_f1 = float(val_macro_f1)
        best_epoch = epoch
        epochs_without_improvement = 0
        best_state = copy.deepcopy(model.state_dict())

        # Save the best model selected using validation data only.
        torch.save(
            {
                "model_state_dict": best_state,
                "class_names": CLASS_NAMES,
                "architecture": "CompactMultiFeatureCNN",
                "trainable_parameters": trainable_params,
                "sample_rate": SAMPLE_RATE,
                "duration_seconds": DURATION_SECONDS,
                "feature_shapes": {
                    "logmel": [64, 157],
                    "mfcc": [13, 157],
                    "chroma": [12, 157],
                },
                "normalization_config_path": str(
                    NORMALIZED_V2_ROOT / "normalization_config.json"
                ),
                "best_epoch": best_epoch,
                "best_validation_macro_f1_supported_classes": (
                    best_val_macro_f1
                ),
                "seed": SEED,
            },
            checkpoint_path,
        )
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_accuracy:.3f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_accuracy:.3f} | "
        f"val_macroF1={val_macro_f1:.3f} | "
        f"best_epoch={best_epoch}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping after {PATIENCE} epochs without improvement.")
        break

# Restore best validation-selected parameters.
if best_state is None:
    raise RuntimeError("Training did not produce a valid checkpoint.")

model.load_state_dict(best_state)
model.eval()

history_path.write_text(json.dumps(history, indent=2))

print("\n=== TRAINING COMPLETE ===")
print("Best epoch:", best_epoch)
print(f"Best validation macro-F1 (supported classes): {best_val_macro_f1:.4f}")
print("Checkpoint:", checkpoint_path)
print("History:", history_path)
print("Test sets evaluated: No")


Epoch 001/100 | train_loss=2.0412 | train_acc=0.178 | val_loss=2.0739 | val_acc=0.038 | val_macroF1=0.015 | best_epoch=1
Epoch 002/100 | train_loss=1.8878 | train_acc=0.593 | val_loss=1.9126 | val_acc=0.588 | val_macroF1=0.226 | best_epoch=2
Epoch 003/100 | train_loss=1.6998 | train_acc=0.659 | val_loss=1.7319 | val_acc=0.626 | val_macroF1=0.271 | best_epoch=3
Epoch 004/100 | train_loss=1.5563 | train_acc=0.681 | val_loss=1.5839 | val_acc=0.687 | val_macroF1=0.314 | best_epoch=4
Epoch 005/100 | train_loss=1.3783 | train_acc=0.725 | val_loss=1.5886 | val_acc=0.626 | val_macroF1=0.325 | best_epoch=5
Epoch 006/100 | train_loss=1.3163 | train_acc=0.733 | val_loss=1.3674 | val_acc=0.687 | val_macroF1=0.342 | best_epoch=6
Epoch 007/100 | train_loss=1.1771 | train_acc=0.733 | val_loss=1.2349 | val_acc=0.679 | val_macroF1=0.349 | best_epoch=7
Epoch 008/100 | train_loss=1.0578 | train_acc=0.745 | val_loss=1.5302 | val_acc=0.527 | val_macroF1=0.255 | best_epoch=7
Epoch 009/100 | train_loss=1.040

In [24]:

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Load the best checkpoint selected using validation data.
checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


def evaluate_saved_model(split_name):
    dataset = make_dataset(split_name)
    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        num_workers=0,
    )

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for logmel, mfcc, chroma, labels in loader:
            logits = model(
                logmel.to(DEVICE),
                mfcc.to(DEVICE),
                chroma.to(DEVICE),
            )

            all_predictions.extend(
                logits.argmax(dim=1).cpu().numpy()
            )
            all_targets.extend(labels.numpy())

    y_true = np.asarray(all_targets)
    y_pred = np.asarray(all_predictions)

    accuracy = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=np.arange(len(CLASS_NAMES)),
        average="macro",
        zero_division=0,
    )

    print("\n" + "=" * 70)
    print(f"TEST RESULTS: {split_name.upper()}")
    print("=" * 70)
    print("Recordings:", len(y_true))
    print(f"Accuracy:          {accuracy:.4f}")
    print(f"Balanced accuracy: {balanced_acc:.4f}")
    print(f"Macro-F1 (8 classes): {macro_f1:.4f}")

    print("\nPer-class classification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=np.arange(len(CLASS_NAMES)),
            target_names=CLASS_NAMES,
            zero_division=0,
            digits=4,
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(len(CLASS_NAMES)),
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual: {name}" for name in CLASS_NAMES],
        columns=[f"Predicted: {name}" for name in CLASS_NAMES],
    )

    print("Confusion matrix:")
    display(cm_df)

    return {
        "split": split_name,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "macro_f1": macro_f1,
        "y_true": y_true,
        "y_pred": y_pred,
        "confusion_matrix": cm,
    }


# Evaluate each held-out test set separately.
icbhi_results = evaluate_saved_model("icbhi_test")
fraiwan_results = evaluate_saved_model("fraiwan_test")



TEST RESULTS: ICBHI_TEST
Recordings: 394
Accuracy:          0.7640
Balanced accuracy: 0.5868
Macro-F1 (8 classes): 0.3363

Per-class classification report:
                precision    recall  f1-score   support

        Asthma     0.0000    0.0000    0.0000         0
Bronchiectasis     0.3333    1.0000    0.5000         2
 Bronchiolitis     1.0000    0.5000    0.6667         6
          COPD     0.9897    0.7956    0.8821       362
       Healthy     0.4000    0.3529    0.3750        17
          LRTI     0.0000    0.0000    0.0000         0
     Pneumonia     0.0000    0.0000    0.0000         0
          URTI     0.2500    0.2857    0.2667         7

      accuracy                         0.7640       394
     macro avg     0.3716    0.3668    0.3363       394
  weighted avg     0.9479    0.7640    0.8440       394

Confusion matrix:


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,Predicted: Asthma,Predicted: Bronchiectasis,Predicted: Bronchiolitis,Predicted: COPD,Predicted: Healthy,Predicted: LRTI,Predicted: Pneumonia,Predicted: URTI
Actual: Asthma,0,0,0,0,0,0,0,0
Actual: Bronchiectasis,0,2,0,0,0,0,0,0
Actual: Bronchiolitis,0,0,3,0,1,0,0,2
Actual: COPD,57,2,0,288,5,0,10,0
Actual: Healthy,2,1,0,2,6,0,2,4
Actual: LRTI,0,0,0,0,0,0,0,0
Actual: Pneumonia,0,0,0,0,0,0,0,0
Actual: URTI,0,1,0,1,3,0,0,2



TEST RESULTS: FRAIWAN_TEST
Recordings: 28
Accuracy:          0.6429
Balanced accuracy: 0.3158
Macro-F1 (8 classes): 0.0978

Per-class classification report:
                precision    recall  f1-score   support

        Asthma     0.6667    0.9474    0.7826        19
Bronchiectasis     0.0000    0.0000    0.0000         0
 Bronchiolitis     0.0000    0.0000    0.0000         0
          COPD     0.0000    0.0000    0.0000         6
       Healthy     0.0000    0.0000    0.0000         0
          LRTI     0.0000    0.0000    0.0000         0
     Pneumonia     0.0000    0.0000    0.0000         3
          URTI     0.0000    0.0000    0.0000         0

      accuracy                         0.6429        28
     macro avg     0.0833    0.1184    0.0978        28
  weighted avg     0.4524    0.6429    0.5311        28

Confusion matrix:


,Predicted: Asthma,Predicted: Bronchiectasis,Predicted: Bronchiolitis,Predicted: COPD,Predicted: Healthy,Predicted: LRTI,Predicted: Pneumonia,Predicted: URTI
Actual: Asthma,18,0,0,0,0,0,1,0
Actual: Bronchiectasis,0,0,0,0,0,0,0,0
Actual: Bronchiolitis,0,0,0,0,0,0,0,0
Actual: COPD,6,0,0,0,0,0,0,0
Actual: Healthy,0,0,0,0,0,0,0,0
Actual: LRTI,0,0,0,0,0,0,0,0
Actual: Pneumonia,3,0,0,0,0,0,0,0
Actual: URTI,0,0,0,0,0,0,0,0


In [25]:

# Audit the saved manifests and feature distributions by dataset source.
import numpy as np
import pandas as pd

audit_splits = {
    "train": loaded_features["train"]["metadata"],
    "validation": loaded_features["validation"]["metadata"],
    "icbhi_test": loaded_features["icbhi_test"]["metadata"],
    "fraiwan_test": loaded_features["fraiwan_test"]["metadata"],
}

for split_name, frame in audit_splits.items():
    print(f"\n{'=' * 65}")
    print(f"{split_name.upper()}: SOURCE COUNTS")
    print("=" * 65)

    print(frame["source_dataset"].value_counts(dropna=False).to_string())

    print(f"\n{split_name.upper()}: SOURCE × DIAGNOSIS")
    print(
        pd.crosstab(
            frame["source_dataset"],
            frame["diagnosis"],
            dropna=False,
        ).reindex(columns=CLASS_NAMES, fill_value=0).to_string()
    )

print("\n" + "=" * 65)
print("FEATURE STATISTICS BY SOURCE AND SPLIT")
print("=" * 65)

feature_names = ["logmel", "mfcc", "chroma"]

for split_name in ["train", "validation", "icbhi_test", "fraiwan_test"]:
    frame = loaded_features[split_name]["metadata"].reset_index(drop=True)

    for source in sorted(frame["source_dataset"].dropna().unique()):
        indices = np.flatnonzero(
            frame["source_dataset"].to_numpy() == source
        )

        print(f"\n{split_name} | source={source} | n={len(indices)}")

        for feature_name in feature_names:
            arr = loaded_features[split_name][feature_name][indices]
            print(
                f"  {feature_name:8s} "
                f"mean={arr.mean():8.4f} "
                f"std={arr.std():8.4f}"
            )



TRAIN: SOURCE COUNTS
source_dataset
ICBHI      427
Fraiwan     74

TRAIN: SOURCE × DIAGNOSIS
diagnosis       Asthma  Bronchiectasis  Bronchiolitis  COPD  Healthy  LRTI  Pneumonia  URTI
source_dataset                                                                             
Fraiwan             47               0              0    15        0     0         12     0
ICBHI                1              13              7   351       12     1         29    13

VALIDATION: SOURCE COUNTS
source_dataset
ICBHI      98
Fraiwan    33

VALIDATION: SOURCE × DIAGNOSIS
diagnosis       Asthma  Bronchiectasis  Bronchiolitis  COPD  Healthy  LRTI  Pneumonia  URTI
source_dataset                                                                             
Fraiwan             27               0              0     6        0     0          0     0
ICBHI                0               1              0    80        6     1          7     3

ICBHI_TEST: SOURCE COUNTS
source_dataset
ICBHI    394

ICBHI_TEST: 

In [26]:

# Cell 22 — Inspect diagnosis provenance and original manifest metadata
from pathlib import Path
import pandas as pd

manifest_paths = {
    "train": OUTPUT_ROOT / "c2_train_manifest.csv",
    "validation": OUTPUT_ROOT / "c2_validation_manifest.csv",
    "ICBHI_test": OUTPUT_ROOT / "experiment_b_icbhi_test_manifest.csv",
    "Fraiwan_test": OUTPUT_ROOT / "experiment_b_fraiwan_test_manifest.csv",
}

for split_name, path in manifest_paths.items():
    print("\n" + "=" * 70)
    print(f"{split_name}: {path}")
    print("=" * 70)

    if not path.exists():
        print("Manifest not found at this path.")
        continue

    df = pd.read_csv(path)
    print("Columns:", df.columns.tolist())
    print("\nSample rows:")
    print(df.head(5).to_string(index=False))

    print("\nDiagnosis values:")
    print(df["diagnosis"].value_counts(dropna=False).to_string())

    if "source_dataset" in df.columns and "diagnosis" in df.columns:
        print("\nSource × diagnosis:")
        print(pd.crosstab(df["source_dataset"], df["diagnosis"]).to_string())



train: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/c2_train_manifest.csv
Columns: ['source_dataset', 'source_recording_id', 'patient_id', 'diagnosis', 'split', 'recording_path']

Sample rows:
source_dataset    source_recording_id patient_id diagnosis split                                                                               recording_path
         ICBHI 103_2b2_Ar_mc_LittC2SE  ICBHI_103    Asthma train /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/103_2b2_Ar_mc_LittC2SE.wav
         ICBHI 106_2b1_Pl_mc_LittC2SE  ICBHI_106      COPD train /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/106_2b1_Pl_mc_LittC2SE.wav
         ICBHI 106_2b1_Pr_mc_LittC2SE  ICBHI_106      COPD train /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/106_2b1_Pr_mc_LittC2SE.wav
         ICBHI 107_2b3_Al_mc_AKGC417L  ICBHI_107      COPD trai

In [27]:

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

search_roots = [
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data",
]

target_names = {
    "experiment_b_icbhi_test_manifest.csv",
    "experiment_b_fraiwan_test_manifest.csv",
    "icbhi_test_manifest.csv",
    "fraiwan_test_manifest.csv",
}

print("Project root:", PROJECT_ROOT)
print("\nSearching for test manifests...")

found_paths = []

for root in search_roots:
    if not root.exists():
        print("Search directory does not exist:", root)
        continue

    for path in root.rglob("*.csv"):
        if path.name in target_names:
            found_paths.append(path)

found_paths = sorted(set(found_paths))

if found_paths:
    for path in found_paths:
        print(path)
else:
    print("No matching test manifests found under the search directories.")


Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification

Searching for test manifests...
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_fraiwan_test_manifest.csv
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_icbhi_test_manifest.csv


In [28]:

import pandas as pd
from pathlib import Path

BASE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
)

test_manifest_paths = {
    "ICBHI_test": BASE_DIR / "experiment_b_icbhi_test_manifest.csv",
    "Fraiwan_test": BASE_DIR / "experiment_b_fraiwan_test_manifest.csv",
}

train_df = pd.read_csv(OUTPUT_ROOT / "c2_train_manifest.csv")
val_df = pd.read_csv(OUTPUT_ROOT / "c2_validation_manifest.csv")

train_patients = set(train_df["patient_id"].astype(str))
val_patients = set(val_df["patient_id"].astype(str))

print("Training patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Train/validation patient overlap:", len(train_patients & val_patients))

for split_name, path in test_manifest_paths.items():
    if not path.exists():
        raise FileNotFoundError(path)

    test_df = pd.read_csv(path)

    print("\n" + "=" * 65)
    print(split_name)
    print("=" * 65)
    print("Manifest:", path)
    print("Rows:", len(test_df))
    print("Missing diagnosis:", int(test_df["diagnosis"].isna().sum()))
    print("Missing patient ID:", int(test_df["patient_id"].isna().sum()))
    print("Duplicate recording IDs:",
          int(test_df["source_recording_id"].duplicated().sum()))

    test_patients = set(test_df["patient_id"].astype(str))

    print("Patient overlap with train:", len(test_patients & train_patients))
    print("Patient overlap with validation:", len(test_patients & val_patients))

    print("\nDiagnosis support:")
    print(test_df["diagnosis"].value_counts().reindex(CLASS_NAMES, fill_value=0).to_string())

    print("\nAudio paths exist:",
          bool(test_df["recording_path"].map(lambda p: Path(p).is_file()).all()))


Training patients: 90
Validation patients: 23
Train/validation patient overlap: 0

ICBHI_test
Manifest: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_icbhi_test_manifest.csv
Rows: 394
Missing diagnosis: 0
Missing patient ID: 0
Duplicate recording IDs: 0
Patient overlap with train: 0
Patient overlap with validation: 0

Diagnosis support:
diagnosis
Asthma              0
Bronchiectasis      2
Bronchiolitis       6
COPD              362
Healthy            17
LRTI                0
Pneumonia           0
URTI                7

Audio paths exist: True

Fraiwan_test
Manifest: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_fraiwan_test_manifest.csv
Rows: 28
Missing diagnosis: 0
Missing patient ID: 0
Duplicate recording IDs: 0
Patient overlap with train: 0
Patient overlap with validation: 0

Diagnosis support:
diagnosis
Asthma            

In [29]:

import pandas as pd
import numpy as np

from pathlib import Path

train_path = OUTPUT_ROOT / "c2_train_manifest.csv"
val_path = OUTPUT_ROOT / "c2_validation_manifest.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)

combined_df = pd.concat(
    [
        train_df.assign(audit_split="train"),
        val_df.assign(audit_split="validation"),
    ],
    ignore_index=True,
)

print("=" * 75)
print("COMBINED ICBHI + FRAIWAN DATASET AUDIT")
print("=" * 75)

print("Total recordings:", len(combined_df))
print("Training recordings:", len(train_df))
print("Validation recordings:", len(val_df))
print("Unique patients:", combined_df["patient_id"].nunique())
print("Unique recording IDs:", combined_df["source_recording_id"].nunique())
print("Duplicate recording IDs:", int(
    combined_df["source_recording_id"].duplicated().sum()
))
print("Missing diagnoses:", int(combined_df["diagnosis"].isna().sum()))
print("Missing source labels:", int(combined_df["source_dataset"].isna().sum()))

print("\n--- SOURCE DISTRIBUTION ---")
print(pd.crosstab(
    combined_df["audit_split"],
    combined_df["source_dataset"],
    margins=True,
).to_string())

print("\n--- CLASS SUPPORT BY SPLIT ---")
print(pd.crosstab(
    combined_df["audit_split"],
    combined_df["diagnosis"],
).reindex(columns=CLASS_NAMES, fill_value=0).to_string())

print("\n--- CLASS SUPPORT BY SOURCE ---")
print(pd.crosstab(
    combined_df["source_dataset"],
    combined_df["diagnosis"],
    margins=True,
).reindex(columns=CLASS_NAMES, fill_value=0).to_string())

print("\n--- CLASS SUPPORT BY SOURCE AND SPLIT ---")
print(pd.crosstab(
    [combined_df["audit_split"], combined_df["source_dataset"]],
    combined_df["diagnosis"],
).reindex(columns=CLASS_NAMES, fill_value=0).to_string())

print("\n--- PATIENT OVERLAP ---")
train_patients = set(train_df["patient_id"].astype(str))
val_patients = set(val_df["patient_id"].astype(str))

print("Train patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Patient overlap:", len(train_patients & val_patients))

print("\n--- DIAGNOSIS COUNTS BY PATIENT ---")
patient_class_counts = (
    combined_df.groupby(["source_dataset", "diagnosis"])["patient_id"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=CLASS_NAMES, fill_value=0)
)
print(patient_class_counts.to_string())


COMBINED ICBHI + FRAIWAN DATASET AUDIT
Total recordings: 632
Training recordings: 501
Validation recordings: 131
Unique patients: 113
Unique recording IDs: 632
Duplicate recording IDs: 0
Missing diagnoses: 0
Missing source labels: 0

--- SOURCE DISTRIBUTION ---
source_dataset  Fraiwan  ICBHI  All
audit_split                        
train                74    427  501
validation           33     98  131
All                 107    525  632

--- CLASS SUPPORT BY SPLIT ---
diagnosis    Asthma  Bronchiectasis  Bronchiolitis  COPD  Healthy  LRTI  Pneumonia  URTI
audit_split                                                                             
train            48              13              7   366       12     1         41    13
validation       27               1              0    86        6     1          7     3

--- CLASS SUPPORT BY SOURCE ---
diagnosis       Asthma  Bronchiectasis  Bronchiolitis  COPD  Healthy  LRTI  Pneumonia  URTI
source_dataset                               

In [30]:

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

search_roots = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "Notebooks 2",
]

keywords = (
    "diagnos",
    "label",
    "metadata",
    "manifest",
    "patient",
    "icbhi",
    "fraiwan",
)

extensions = {".csv", ".tsv", ".xlsx", ".json", ".txt", ".parquet"}

candidates = set()

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in extensions:
            continue

        name = path.name.lower()
        if any(keyword in name for keyword in keywords):
            candidates.add(path)

print("Potential diagnosis/metadata files:")
for path in sorted(candidates):
    print(path.relative_to(PROJECT_ROOT))


Potential diagnosis/metadata files:
data/interim/.ipynb_checkpoints/patients-checkpoint.csv
data/interim/patients.csv
data/processed/deep_learning/cnn_baseline/cnn_patient_disjoint_manifest.csv
data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv
data/processed/deep_learning/cnn_baseline/train_cnn_manifest.csv
data/processed/deep_learning/cnn_baseline/validation_cnn_manifest.csv
data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict/evaluation/fraiwan_external_test_classification_report.csv
data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict/evaluation/fraiwan_external_test_confusion_matrix.csv
data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict/evaluation/fraiwan_external_test_recording_predictions.csv
data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experi

In [31]:

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

search_roots = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "Notebooks 2",
]

keywords = (
    "diagnos", "label", "metadata",
    "manifest", "patient", "icbhi", "fraiwan"
)

extensions = {".csv", ".tsv", ".xlsx", ".json", ".txt", ".parquet"}

candidates = set()

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in extensions:
            if any(k in path.name.lower() for k in keywords):
                candidates.add(path)

print("Potential diagnosis/metadata files:")
for path in sorted(candidates):
    print(path.relative_to(PROJECT_ROOT))


Potential diagnosis/metadata files:
data/interim/.ipynb_checkpoints/patients-checkpoint.csv
data/interim/patients.csv
data/processed/deep_learning/cnn_baseline/cnn_patient_disjoint_manifest.csv
data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv
data/processed/deep_learning/cnn_baseline/train_cnn_manifest.csv
data/processed/deep_learning/cnn_baseline/validation_cnn_manifest.csv
data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict/evaluation/fraiwan_external_test_classification_report.csv
data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict/evaluation/fraiwan_external_test_confusion_matrix.csv
data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict/evaluation/fraiwan_external_test_recording_predictions.csv
data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experi

In [32]:

import json
import numpy as np
from pathlib import Path

MULTIFEATURE_V2_ROOT = OUTPUT_ROOT / "features_multifeature_v2"
NORMALIZED_V2_ROOT = OUTPUT_ROOT / "features_multifeature_v2_normalized"
NORMALIZED_V2_ROOT.mkdir(parents=True, exist_ok=True)

FEATURE_NAMES = ["logmel", "mfcc", "chroma"]
SPLIT_NAMES = ["train", "validation", "icbhi_test", "fraiwan_test"]

# Load the corrected split's cached features.
raw_features_v2 = {}

for split in SPLIT_NAMES:
    path = MULTIFEATURE_V2_ROOT / f"{split}_features.npz"

    if not path.exists():
        raise FileNotFoundError(f"Missing feature file: {path}")

    with np.load(path) as data:
        raw_features_v2[split] = {
            name: data[name].astype(np.float32)
            for name in FEATURE_NAMES
        }

# Calculate normalization statistics from TRAINING data only.
normalization_stats_v2 = {}

for name in FEATURE_NAMES:
    arr = raw_features_v2["train"][name]
    mean = float(arr.mean())
    std = float(arr.std())

    if not np.isfinite(mean) or not np.isfinite(std) or std <= 0:
        raise ValueError(f"Invalid normalization statistics for {name}")

    normalization_stats_v2[name] = {
        "mean": mean,
        "std": std
    }

print("Training-only normalization statistics:")
for name, stats in normalization_stats_v2.items():
    print(f"{name}: mean={stats['mean']:.6f}, std={stats['std']:.6f}")

# Normalize every split using the SAME training statistics.
for split in SPLIT_NAMES:
    normalized = {}

    for name in FEATURE_NAMES:
        stats = normalization_stats_v2[name]
        arr = raw_features_v2[split][name]

        normalized[name] = (
            (arr - stats["mean"]) / stats["std"]
        ).astype(np.float32)

    out_path = NORMALIZED_V2_ROOT / f"{split}_features_normalized.npz"

    # Never silently overwrite previous results.
    if out_path.exists():
        raise FileExistsError(
            f"Output already exists: {out_path}\n"
            "Inspect it before deciding whether to reuse or replace it."
        )

    np.savez_compressed(out_path, **normalized)
    print(f"Saved {split}: {out_path}")

# Save the normalization parameters for reproducibility and deployment.
config_path = NORMALIZED_V2_ROOT / "normalization_config.json"

if config_path.exists():
    raise FileExistsError(f"Configuration already exists: {config_path}")

config_path.write_text(
    json.dumps({
        "feature_names": FEATURE_NAMES,
        "training_statistics": normalization_stats_v2,
        "method": "global standardization using training data only"
    }, indent=2)
)

print("\nNormalization completed successfully.")


Training-only normalization statistics:
logmel: mean=-38.733738, std=18.971832
mfcc: mean=-5.886697, std=95.320946
chroma: mean=0.707414, std=0.223571


FileExistsError: Output already exists: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/features_multifeature_v2_normalized/train_features_normalized.npz
Inspect it before deciding whether to reuse or replace it.

In [33]:

from pathlib import Path
import numpy as np
import json

NORMALIZED_V2_ROOT = OUTPUT_ROOT / "features_multifeature_v2_normalized"

expected_files = [
    "train_features_normalized.npz",
    "validation_features_normalized.npz",
    "icbhi_test_features_normalized.npz",
    "fraiwan_test_features_normalized.npz",
    "normalization_config.json",
]

print("Existing normalized feature files:\n")

for filename in expected_files:
    path = NORMALIZED_V2_ROOT / filename
    print(f"{filename}: {'EXISTS' if path.exists() else 'MISSING'}")

    if path.exists() and path.suffix == ".npz":
        with np.load(path) as data:
            for feature in ["logmel", "mfcc", "chroma"]:
                if feature in data:
                    arr = data[feature]
                    print(
                        f"  {feature}: shape={arr.shape}, "
                        f"dtype={arr.dtype}, "
                        f"finite={np.isfinite(arr).all()}"
                    )

config_path = NORMALIZED_V2_ROOT / "normalization_config.json"

if config_path.exists():
    print("\nExisting normalization configuration:")
    print(config_path.read_text())


Existing normalized feature files:

train_features_normalized.npz: EXISTS
  logmel: shape=(501, 64, 157), dtype=float32, finite=True
  mfcc: shape=(501, 13, 157), dtype=float32, finite=True
  chroma: shape=(501, 12, 157), dtype=float32, finite=True
validation_features_normalized.npz: EXISTS
  logmel: shape=(131, 64, 157), dtype=float32, finite=True
  mfcc: shape=(131, 13, 157), dtype=float32, finite=True
  chroma: shape=(131, 12, 157), dtype=float32, finite=True
icbhi_test_features_normalized.npz: EXISTS
  logmel: shape=(394, 64, 157), dtype=float32, finite=True
  mfcc: shape=(394, 13, 157), dtype=float32, finite=True
  chroma: shape=(394, 12, 157), dtype=float32, finite=True
fraiwan_test_features_normalized.npz: EXISTS
  logmel: shape=(28, 64, 157), dtype=float32, finite=True
  mfcc: shape=(28, 13, 157), dtype=float32, finite=True
  chroma: shape=(28, 12, 157), dtype=float32, finite=True
normalization_config.json: EXISTS

Existing normalization configuration:
{
  "method": "global sta

In [34]:

import numpy as np
import pandas as pd
from pathlib import Path

NORMALIZED_V2_ROOT = OUTPUT_ROOT / "features_multifeature_v2_normalized"

# Load the corrected manifests.
train_manifest = pd.read_csv(OUTPUT_ROOT / "c2_train_manifest.csv")
val_manifest = pd.read_csv(OUTPUT_ROOT / "c2_validation_manifest.csv")

icbhi_manifest = pd.read_csv(BASE_DIR / "experiment_b_icbhi_test_manifest.csv")
fraiwan_manifest = pd.read_csv(BASE_DIR / "experiment_b_fraiwan_test_manifest.csv")

split_info = {
    "train": (train_manifest, "train_features_normalized.npz"),
    "validation": (val_manifest, "validation_features_normalized.npz"),
    "icbhi_test": (icbhi_manifest, "icbhi_test_features_normalized.npz"),
    "fraiwan_test": (fraiwan_manifest, "fraiwan_test_features_normalized.npz"),
}

for split, (manifest, filename) in split_info.items():
    feature_path = NORMALIZED_V2_ROOT / filename

    with np.load(feature_path) as data:
        n_features = len(data["logmel"])

        assert len(data["mfcc"]) == n_features
        assert len(data["chroma"]) == n_features
        assert n_features == len(manifest), (
            f"{split}: {n_features} feature rows but "
            f"{len(manifest)} manifest rows"
        )

        assert all(
            np.isfinite(data[name]).all()
            for name in ["logmel", "mfcc", "chroma"]
        ), f"{split}: non-finite feature values found"

    assert manifest["diagnosis"].notna().all(), (
        f"{split}: missing diagnosis labels"
    )

    unknown = sorted(set(manifest["diagnosis"]) - set(CLASS_TO_IDX))
    assert not unknown, f"{split}: unexpected labels: {unknown}"

    print(
        f"{split}: PASS | recordings={len(manifest)} | "
        f"classes={manifest['diagnosis'].nunique()}"
    )

print("\nTraining class distribution:")
print(train_manifest["diagnosis"].value_counts().reindex(CLASS_NAMES, fill_value=0))

print("\nValidation class distribution:")
print(val_manifest["diagnosis"].value_counts().reindex(CLASS_NAMES, fill_value=0))

print("\nFeature and label row-count checks passed.")


train: PASS | recordings=501 | classes=8
validation: PASS | recordings=131 | classes=7
icbhi_test: PASS | recordings=394 | classes=5
fraiwan_test: PASS | recordings=28 | classes=3

Training class distribution:
diagnosis
Asthma             48
Bronchiectasis     13
Bronchiolitis       7
COPD              366
Healthy            12
LRTI                1
Pneumonia          41
URTI               13
Name: count, dtype: int64

Validation class distribution:
diagnosis
Asthma            27
Bronchiectasis     1
Bronchiolitis      0
COPD              86
Healthy            6
LRTI               1
Pneumonia          7
URTI               3
Name: count, dtype: int64

Feature and label row-count checks passed.


In [35]:

import pandas as pd
from pathlib import Path

MULTIFEATURE_V2_ROOT = OUTPUT_ROOT / "features_multifeature_v2"

metadata_files = {
    "train": MULTIFEATURE_V2_ROOT / "train_metadata.csv",
    "validation": MULTIFEATURE_V2_ROOT / "validation_metadata.csv",
    "icbhi_test": MULTIFEATURE_V2_ROOT / "icbhi_test_metadata.csv",
    "fraiwan_test": MULTIFEATURE_V2_ROOT / "fraiwan_test_metadata.csv",
}

manifests = {
    "train": train_manifest,
    "validation": val_manifest,
    "icbhi_test": icbhi_manifest,
    "fraiwan_test": fraiwan_manifest,
}

for split, metadata_path in metadata_files.items():
    print(f"\n{'=' * 18} {split.upper()} {'=' * 18}")

    if not metadata_path.exists():
        print(f"Metadata file not found: {metadata_path}")
        continue

    metadata = pd.read_csv(metadata_path)
    manifest = manifests[split]

    print("Metadata columns:", metadata.columns.tolist())
    print("Manifest columns:", manifest.columns.tolist())
    print("\nFeature metadata preview:")
    print(metadata.head(3).to_string(index=False))

    # Find shared recording-identity columns.
    identity_candidates = [
        "source_recording_id",
        "recording_id",
        "recording",
        "filename",
        "file_name",
        "recording_path",
    ]

    shared = [
        col for col in identity_candidates
        if col in metadata.columns and col in manifest.columns
    ]

    if len(metadata) != len(manifest):
        print(
            f"ROW COUNT MISMATCH: metadata={len(metadata)}, "
            f"manifest={len(manifest)}"
        )
    elif not shared:
        print("No shared recording-ID column found; manual inspection needed.")
    else:
        for col in shared:
            same_order = (
                metadata[col].astype(str).reset_index(drop=True).equals(
                    manifest[col].astype(str).reset_index(drop=True)
                )
            )
            print(f"{col} matches in exact row order: {same_order}")

        if "diagnosis" in metadata.columns:
            same_labels = (
                metadata["diagnosis"].astype(str).reset_index(drop=True).equals(
                    manifest["diagnosis"].astype(str).reset_index(drop=True)
                )
            )
            print(f"Diagnosis labels match in exact row order: {same_labels}")



================== TRAIN ==================
Metadata columns: ['source_dataset', 'source_recording_id', 'patient_id', 'diagnosis', 'recording_path']
Manifest columns: ['source_dataset', 'source_recording_id', 'patient_id', 'diagnosis', 'split', 'recording_path']

Feature metadata preview:
source_dataset    source_recording_id patient_id diagnosis                                                                               recording_path
         ICBHI 103_2b2_Ar_mc_LittC2SE  ICBHI_103    Asthma /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/103_2b2_Ar_mc_LittC2SE.wav
         ICBHI 106_2b1_Pl_mc_LittC2SE  ICBHI_106      COPD /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/106_2b1_Pl_mc_LittC2SE.wav
         ICBHI 106_2b1_Pr_mc_LittC2SE  ICBHI_106      COPD /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/106_2b1_Pr_mc_LittC2SE.wav
source_recording_id matches in exact row order: True
recording_path matches in exact row order: True


In [36]:

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight

NORMALIZED_V2_ROOT = OUTPUT_ROOT / "features_multifeature_v2_normalized"

SPLIT_FILES = {
    "train": "train_features_normalized.npz",
    "validation": "validation_features_normalized.npz",
    "icbhi_test": "icbhi_test_features_normalized.npz",
    "fraiwan_test": "fraiwan_test_features_normalized.npz",
}

SPLIT_MANIFESTS = {
    "train": train_manifest,
    "validation": val_manifest,
    "icbhi_test": icbhi_manifest,
    "fraiwan_test": fraiwan_manifest,
}

features = {}
labels = {}

for split, filename in SPLIT_FILES.items():
    with np.load(NORMALIZED_V2_ROOT / filename) as data:
        features[split] = {
            name: np.array(data[name], dtype=np.float32, copy=True)
            for name in ["logmel", "mfcc", "chroma"]
        }

    manifest = SPLIT_MANIFESTS[split]
    labels[split] = np.array(
        [CLASS_TO_IDX[label] for label in manifest["diagnosis"]],
        dtype=np.int64
    )

    assert len(labels[split]) == len(features[split]["logmel"])
    print(f"{split}: {len(labels[split])} recordings loaded")

# Compute class weights from TRAINING labels only.
present_classes = np.unique(labels["train"])

raw_weights = compute_class_weight(
    class_weight="balanced",
    classes=present_classes,
    y=labels["train"]
)

class_weights = np.zeros(len(CLASS_NAMES), dtype=np.float32)
class_weights[present_classes] = raw_weights

# Cap extreme weights to reduce instability for very rare classes.
class_weights = np.minimum(class_weights, 4.0)
class_weights[present_classes] /= class_weights[present_classes].mean()

print("\nClass weights:")
for name, weight in zip(CLASS_NAMES, class_weights):
    print(f"{name:18s}: {weight:.4f}")

# Create training and validation tensors.
def make_dataset(split):
    return TensorDataset(
        torch.from_numpy(features[split]["logmel"].copy()),
        torch.from_numpy(features[split]["mfcc"].copy()),
        torch.from_numpy(features[split]["chroma"].copy()),
        torch.from_numpy(labels[split].copy())
    )

train_dataset = make_dataset("train")
validation_dataset = make_dataset("validation")

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("\nDevice:", DEVICE)
print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Training preparation complete.")


train: 501 recordings loaded
validation: 131 recordings loaded
icbhi_test: 394 recordings loaded
fraiwan_test: 28 recordings loaded

Class weights:
Asthma            : 0.4537
Bronchiectasis    : 1.3911
Bronchiolitis     : 1.3911
COPD              : 0.0595
Healthy           : 1.3911
LRTI              : 1.3911
Pneumonia         : 0.5312
URTI              : 1.3911

Device: mps
Training batches: 16
Validation batches: 5
Training preparation complete.


In [38]:

from pathlib import Path
import torch

BEST_MODEL_PATH = OUTPUT_ROOT / "c2_compact_multifeature_best.pt"

print("Checkpoint exists:", BEST_MODEL_PATH.exists())
print("Checkpoint size (MB):", round(BEST_MODEL_PATH.stat().st_size / (1024**2), 3))

try:
    checkpoint = torch.load(
        BEST_MODEL_PATH,
        map_location="cpu",
        weights_only=True
    )

    print("\nCheckpoint type:", type(checkpoint).__name__)

    if isinstance(checkpoint, dict):
        print("Top-level keys:", list(checkpoint.keys()))

        if "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        elif "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        elif all(torch.is_tensor(v) for v in checkpoint.values()):
            state_dict = checkpoint
        else:
            state_dict = None

        if state_dict is not None:
            print("Parameter tensors:", len(state_dict))
            print("First keys:", list(state_dict.keys())[:8])

        for key in ["epoch", "best_val_macro_f1", "val_macro_f1",
                    "model_name", "num_classes"]:
            if key in checkpoint:
                print(f"{key}: {checkpoint[key]}")

except Exception as exc:
    print("Could not inspect checkpoint:", repr(exc))


Checkpoint exists: True
Checkpoint size (MB): 0.098

Checkpoint type: dict
Top-level keys: ['model_state_dict', 'class_names', 'architecture', 'trainable_parameters', 'sample_rate', 'duration_seconds', 'feature_shapes', 'normalization_config_path', 'best_epoch', 'best_validation_macro_f1_supported_classes', 'seed']
Parameter tensors: 58
First keys: ['logmel_branch.features.0.weight', 'logmel_branch.features.1.weight', 'logmel_branch.features.1.bias', 'logmel_branch.features.1.running_mean', 'logmel_branch.features.1.running_var', 'logmel_branch.features.1.num_batches_tracked', 'logmel_branch.features.4.weight', 'logmel_branch.features.5.weight']


In [39]:

import torch

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location="cpu",
    weights_only=True
)

print("Architecture:", checkpoint.get("architecture"))
print("Trainable parameters:", checkpoint.get("trainable_parameters"))
print("Class names:", checkpoint.get("class_names"))
print("Best epoch:", checkpoint.get("best_epoch"))
print(
    "Best validation macro-F1:",
    checkpoint.get("best_validation_macro_f1_supported_classes")
)
print("Random seed:", checkpoint.get("seed"))
print("Sample rate:", checkpoint.get("sample_rate"))
print("Duration (seconds):", checkpoint.get("duration_seconds"))
print("Feature shapes:", checkpoint.get("feature_shapes"))
print(
    "Normalization config:",
    checkpoint.get("normalization_config_path")
)

print("\nExpected class names:", CLASS_NAMES)
print(
    "Class order matches:",
    checkpoint.get("class_names") == CLASS_NAMES
)
print(
    "Parameter count matches:",
    checkpoint.get("trainable_parameters") == 19588
)


Architecture: CompactMultiFeatureCNN
Trainable parameters: 19588
Class names: ['Asthma', 'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'LRTI', 'Pneumonia', 'URTI']
Best epoch: 31
Best validation macro-F1: 0.3914285714285714
Random seed: 42
Sample rate: 4000
Duration (seconds): 5
Feature shapes: {'logmel': [64, 157], 'mfcc': [13, 157], 'chroma': [12, 157]}
Normalization config: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c2_icbhi_fraiwan_high_capacity/features_multifeature_v2_normalized/normalization_config.json

Expected class names: ['Asthma', 'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'LRTI', 'Pneumonia', 'URTI']
Class order matches: True
Parameter count matches: True


In [40]:

import torch

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

# Verify that the checkpoint belongs to the expected experiment.
assert checkpoint["architecture"] == "CompactMultiFeatureCNN"
assert checkpoint["trainable_parameters"] == 19588
assert checkpoint["class_names"] == CLASS_NAMES
assert checkpoint["sample_rate"] == SAMPLE_RATE
assert checkpoint["duration_seconds"] == DURATION_SECONDS

# Load the saved weights into the compact architecture.
model = CompactMultiFeatureCNN(
    num_classes=len(CLASS_NAMES),
    dropout=0.30
).to(DEVICE)

model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

parameter_count = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

# Basic forward-pass verification using one real validation example.
with torch.no_grad():
    x_logmel = torch.from_numpy(
        features["validation"]["logmel"][:1].copy()
    ).to(DEVICE)

    x_mfcc = torch.from_numpy(
        features["validation"]["mfcc"][:1].copy()
    ).to(DEVICE)

    x_chroma = torch.from_numpy(
        features["validation"]["chroma"][:1].copy()
    ).to(DEVICE)

    logits = model(x_logmel, x_mfcc, x_chroma)
    probabilities = torch.softmax(logits, dim=1)

assert logits.shape == (1, len(CLASS_NAMES))
assert torch.isfinite(logits).all()
assert torch.isfinite(probabilities).all()

predicted_index = int(probabilities.argmax(dim=1).item())

print("Checkpoint loaded successfully.")
print("Trainable parameters:", f"{parameter_count:,}")
print("Best saved epoch:", checkpoint["best_epoch"])
print(
    "Best validation macro-F1:",
    round(checkpoint["best_validation_macro_f1_supported_classes"], 4)
)
print("Output shape:", tuple(logits.shape))
print("Predicted class for validation example:", CLASS_NAMES[predicted_index])
print("Probability sum:", round(float(probabilities.sum().item()), 6))


Checkpoint loaded successfully.
Trainable parameters: 19,588
Best saved epoch: 31
Best validation macro-F1: 0.3914
Output shape: (1, 8)
Predicted class for validation example: Bronchiectasis
Probability sum: 1.0


In [42]:

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

def predict_split(split, batch_size=32):
    model.eval()
    predictions = []
    all_probabilities = []

    n = len(labels[split])

    with torch.no_grad():
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)

            x_logmel = torch.from_numpy(
                features[split]["logmel"][start:end].copy()
            ).to(DEVICE)

            x_mfcc = torch.from_numpy(
                features[split]["mfcc"][start:end].copy()
            ).to(DEVICE)

            x_chroma = torch.from_numpy(
                features[split]["chroma"][start:end].copy()
            ).to(DEVICE)

            logits = model(x_logmel, x_mfcc, x_chroma)
            probabilities = torch.softmax(logits, dim=1)

            predictions.extend(
                probabilities.argmax(dim=1).cpu().numpy().tolist()
            )
            all_probabilities.extend(
                probabilities.cpu().numpy().tolist()
            )

    return np.asarray(predictions), np.asarray(all_probabilities)


test_results = {}

for split in ["icbhi_test", "fraiwan_test"]:
    y_true = labels[split]
    y_pred, y_prob = predict_split(split)

    test_results[split] = {
        "y_true": y_true.copy(),
        "y_pred": y_pred.copy(),
        "probabilities": y_prob.copy()
    }

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(range(len(CLASS_NAMES))),
        average="macro",
        zero_division=0
    )

    print("\n" + "=" * 65)
    print(f"{split.upper()} RESULTS")
    print("=" * 65)
    print(f"Recordings: {len(y_true)}")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(
        f"Balanced accuracy: "
        f"{balanced_accuracy_score(y_true, y_pred):.4f}"
    )
    print(f"Macro-F1 (all 8 classes): {macro_f1:.4f}")

    print("\nPer-class classification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=list(range(len(CLASS_NAMES))),
            target_names=CLASS_NAMES,
            zero_division=0,
            digits=4
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(CLASS_NAMES)))
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"True {name}" for name in CLASS_NAMES],
        columns=[f"Pred {name}" for name in CLASS_NAMES]
    )

    print("Confusion matrix (rows=true, columns=predicted):")
    display(cm_df)

print("\nEvaluation complete. The checkpoint was not modified.")



ICBHI_TEST RESULTS
Recordings: 394
Accuracy: 0.7640
Balanced accuracy: 0.5868
Macro-F1 (all 8 classes): 0.3363

Per-class classification report:
                precision    recall  f1-score   support

        Asthma     0.0000    0.0000    0.0000         0
Bronchiectasis     0.3333    1.0000    0.5000         2
 Bronchiolitis     1.0000    0.5000    0.6667         6
          COPD     0.9897    0.7956    0.8821       362
       Healthy     0.4000    0.3529    0.3750        17
          LRTI     0.0000    0.0000    0.0000         0
     Pneumonia     0.0000    0.0000    0.0000         0
          URTI     0.2500    0.2857    0.2667         7

      accuracy                         0.7640       394
     macro avg     0.3716    0.3668    0.3363       394
  weighted avg     0.9479    0.7640    0.8440       394

Confusion matrix (rows=true, columns=predicted):


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,Pred Asthma,Pred Bronchiectasis,Pred Bronchiolitis,Pred COPD,Pred Healthy,Pred LRTI,Pred Pneumonia,Pred URTI
True Asthma,0,0,0,0,0,0,0,0
True Bronchiectasis,0,2,0,0,0,0,0,0
True Bronchiolitis,0,0,3,0,1,0,0,2
True COPD,57,2,0,288,5,0,10,0
True Healthy,2,1,0,2,6,0,2,4
True LRTI,0,0,0,0,0,0,0,0
True Pneumonia,0,0,0,0,0,0,0,0
True URTI,0,1,0,1,3,0,0,2



FRAIWAN_TEST RESULTS
Recordings: 28
Accuracy: 0.6429
Balanced accuracy: 0.3158
Macro-F1 (all 8 classes): 0.0978

Per-class classification report:
                precision    recall  f1-score   support

        Asthma     0.6667    0.9474    0.7826        19
Bronchiectasis     0.0000    0.0000    0.0000         0
 Bronchiolitis     0.0000    0.0000    0.0000         0
          COPD     0.0000    0.0000    0.0000         6
       Healthy     0.0000    0.0000    0.0000         0
          LRTI     0.0000    0.0000    0.0000         0
     Pneumonia     0.0000    0.0000    0.0000         3
          URTI     0.0000    0.0000    0.0000         0

      accuracy                         0.6429        28
     macro avg     0.0833    0.1184    0.0978        28
  weighted avg     0.4524    0.6429    0.5311        28

Confusion matrix (rows=true, columns=predicted):


,Pred Asthma,Pred Bronchiectasis,Pred Bronchiolitis,Pred COPD,Pred Healthy,Pred LRTI,Pred Pneumonia,Pred URTI
True Asthma,18,0,0,0,0,0,1,0
True Bronchiectasis,0,0,0,0,0,0,0,0
True Bronchiolitis,0,0,0,0,0,0,0,0
True COPD,6,0,0,0,0,0,0,0
True Healthy,0,0,0,0,0,0,0,0
True LRTI,0,0,0,0,0,0,0,0
True Pneumonia,3,0,0,0,0,0,0,0
True URTI,0,0,0,0,0,0,0,0



Evaluation complete. The checkpoint was not modified.
